# Maryland Infrastructure Analysis

## Import Statements

In [8]:
import pandas as pd
from bs4 import BeautifulSoup as bs
import lxml
import requests
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

import numpy as np
import math

# !pip install area
from area import area

# !pip install sodapy
from sodapy import Socrata

from sklearn.cluster import KMeans

import itertools

import matplotlib.pyplot as plt 
import matplotlib.cm as cm
import matplotlib.colors as colors
import matplotlib.dates as mdates

import folium
from geopy.geocoders import Nominatim # convert an address into latitude and longitude values

from shapely.geometry import Point
from shapely.geometry.polygon import Polygon

# Import BeautifulSoup
from bs4 import BeautifulSoup

import sys
import os
from pathlib import Path

from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.arima.model import ARIMA

# Add the parent directory (main_directory) to the Python path
sys.path.append(str(Path("..")))

# Import the variable from the keys.py file in the subdirectory2 folder
from constants import * #import private API credentials

from llm_config import LLMManager, ModelLister

print("Imports complete")

Imports complete


## Set Save Parameters

### Set path to save figures and data

In [9]:
from pathlib import Path

current_directory = Path.cwd()
output_directory = current_directory.parent / 'data'
output_directory = output_directory.resolve()

print(output_directory)

C:\Users\ChrisPeoples\OneDrive - Peoples Partners and Associates, LLC\PP&A Working Directory\70_Studies_and_Reports\2023_EV_Infrastructure_Trends\data


### Function to save figures and data

In [10]:
def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
    path = os.path.join(output_directory, fig_id + "." + fig_extension)
    print("Saving figure", fig_id)
    if tight_layout:
        plt.tight_layout()
    plt.savefig(path, format=fig_extension, dpi=resolution)

# Save data to csv
def save_data(data, filename):
    filetype = ".csv"
    filename = filename + filetype
    path = os.path.join(output_directory, filename)
    print("Saving data", filename)
    data.to_csv(path)
    print("File saved at " + path + "/" + filename)

print("Done")

Done


## Data Prep and ETL

### GDP | USA by State
pull GDP data by U.S. state, you can use the Bureau of Economic Analysis (BEA) API. The BEA's API is designed for developers and analysts to search, retrieve, and analyze data.: <a href="https://www.bea.gov/resources/for-developers">BEA Developer Site</a>

In [11]:
def get_data_from_bea(api_key, datasetname, tablename, linecode, year, geofips):
    base_url = 'https://apps.bea.gov/api/data'
    params = {
        'UserID': api_key,
        'method': 'GetData',
        'datasetname': datasetname,
        'tablename': tablename,
        'linecode': linecode,
        'year': year,
        'GeoFips': geofips,
        'ResultFormat': 'json'
    }
    response = requests.get(base_url, params=params)
    data = response.json()
    return data

api_key = BEA_KEY
datasetname = 'Regional'
tablename = 'SASUMMARY'  # GDP by state
linecode = '4'  # Line code for GDP
year = 'ALL'  # Year of the data
geofips = 'STATE'  # GeoFips code for all states

data = get_data_from_bea(api_key, datasetname, tablename, linecode, year, geofips)

# Extract the data part of the JSON
data_values = data['BEAAPI']['Results']['Data']

# Convert the data to a DataFrame
df = pd.DataFrame(data_values)

# Remove commas from the 'DataValue' column
df['DataValue'] = df['DataValue'].str.replace(',', '')

# Convert the 'DataValue' column to numeric
df['DataValue'] = pd.to_numeric(df['DataValue'], errors='coerce')

# List of all U.S. states including District of Columbia
states = ['Alabama', 'Alaska', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Hawaii', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']

# Filter the DataFrame to include only the states
df = df[df['GeoName'].isin(states)]

# Pivot the DataFrame to have 'GeoName' as index, 'TimePeriod' as columns, and 'DataValue' as values
df_pivot = df.pivot(index='GeoName', columns='TimePeriod', values='DataValue')

# Print the pivoted DataFrame
state_gdp = df_pivot

state_gdp.head()

TimePeriod,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022
GeoName,,,,,,,,,,,,,,,,,,,,,,,,,
Alabama,110212.0,115680.1,119851.7,122915.5,127505.0,134152.6,147715.2,158846.8,166469.0,172975.2,174526.3,170930.9,177249.2,183916.6,189245.5,194786.9,197406.9,202372.4,207368.4,214606.3,223859.3,231561.9,230892.1,254109.7,277817.5
Alaska,24227.5,24744.3,26806.6,28494.1,29756.8,32037.9,35302.4,40356.6,45094.1,49583.7,55122.5,49957.8,53331.6,56896.3,58283.6,57247.7,56484.9,51490.9,50727.7,53301.5,54899.6,54728.2,50475.2,57349.4,63618.0
Arizona,143302.6,155755.8,165110.7,171909.6,180522.3,193634.9,206541.1,227915.9,245957.0,261392.0,262926.0,246424.3,251153.0,260915.7,271440.0,278591.6,287666.6,299393.3,313081.4,332001.8,351879.5,372393.5,382072.3,420026.7,458949.8
Arkansas,62396.7,66811.3,68678.5,70616.7,74113.9,78695.1,85199.6,90887.7,95875.1,98381.6,99706.8,97508.1,101486.5,105768.1,108492.1,113227.3,116139.4,117786.8,119152.4,122466.7,127535.7,131578.3,133969.1,148676.1,165220.6
California,1147520.4,1241899.7,1356975.4,1375761.3,1418429.6,1497918.7,1588177.4,1698560.4,1812210.0,1898902.0,1944695.3,1890165.9,1954092.7,2023500.0,2113096.4,2220389.9,2335286.5,2473555.9,2569634.0,2728743.1,2897200.7,3042694.1,3020173.4,3373240.7,3598102.7


In [69]:
# Save dataframe using the save_data function
save_data(state_gdp, 'USA_State_GDP')

### Census Population Data | USA by State
<div>Current census data is from 2019</div>
<div>Data is pulled from <span><a href="https://www.census.gov/data/developers/guidance/api-user-guide.html">Census.gov</a></span></div>

In [13]:
def get_census_data(year):
    # Define the base URL for the API endpoint
    if year == 2019:
        # Define the base URL for the API endpoint
        base_url = f"https://api.census.gov/data/2019/pep/population"

        # Define the parameters for the API request
        params = {
            "get": "NAME,POP",
            "for": "state:*",
            "key": CENSUS_KEY
        }
    else:
        # Define the base URL for the API endpoint
        base_url = f"https://api.census.gov/data/{year}/acs/acs1?get=NAME,B01001_001E&for=state:*"

        # Define the parameters for the API request
        params = {
            # "get": "NAME,b01001_001E",
            # "for": "state:*",
            "key": CENSUS_KEY
        }

    # Send a GET request to the API endpoint
    response = requests.get(base_url, params=params)

    print(response.status_code)
    print(response.text)

    # Convert the response to a DataFrame
    data = response.json()
    df = pd.DataFrame(data[1:], columns=data[0])
    df['Year'] = year  # Add a column for the year

    return df

# Initialize an empty DataFrame to store all the data
hist_population = pd.DataFrame()

# Loop through the years 2010 to 2019
for year in range(2010, 2020):
    # Get the data for the current year
    year_data = get_census_data(year)
    # Append the data for the current year to the all_data DataFrame

    hist_population = pd.concat([hist_population, year_data])

# Reset the index of the all_data DataFrame
hist_population.reset_index(drop=True, inplace=True)

# Replace NaN values in the POP column with values from the B01001_001E column
hist_population['POP'] = hist_population['POP'].fillna(hist_population['B01001_001E'])

# Drop the B01001_001E column
hist_population.drop(columns='B01001_001E', inplace=True)

# Convert POP values to integers
hist_population['POP'] = hist_population['POP'].astype(int)

# Pivot Data to have 'NAME' as index, 'Year' as columns, and 'POP' as values
hist_population = hist_population.pivot(index='NAME', columns='Year', values='POP')

print(hist_population)

200
[["NAME","B01001_001E","state"],
["Alabama","4785298","01"],
["Alaska","713985","02"],
["Arizona","6413737","04"],
["Arkansas","2921606","05"],
["California","37349363","06"],
["Colorado","5049071","08"],
["Connecticut","3577073","09"],
["Delaware","899769","10"],
["District of Columbia","604453","11"],
["Florida","18843326","12"],
["Georgia","9712587","13"],
["Hawaii","1363621","15"],
["Idaho","1571450","16"],
["Illinois","12843166","17"],
["Indiana","6490621","18"],
["Iowa","3049883","19"],
["Kansas","2859169","20"],
["Kentucky","4346266","21"],
["Louisiana","4544228","22"],
["Maine","1327567","23"],
["Maryland","5785982","24"],
["Massachusetts","6557254","25"],
["Michigan","9877574","26"],
["Minnesota","5310584","27"],
["Mississippi","2970036","28"],
["Missouri","5996231","29"],
["Montana","990898","30"],
["Nebraska","1830429","31"],
["Nevada","2704642","32"],
["New Hampshire","1316759","33"],
["New Jersey","8801624","34"],
["New Mexico","2065932","35"],
["New York","19392283","

#### Forecast population from 2020 to 2029 using historic census data

In [14]:
# Assuming your data is in a pandas DataFrame called df
df = hist_population

# Transpose the DataFrame and reset the index
df = df.transpose().reset_index()
# Rename the 'index' column to 'Year'
df = df.rename(columns={'index': 'Year'})

# Convert the DataFrame from wide format to long format
df_long = pd.melt(df, id_vars='Year', var_name='State', value_name='Population')

# # set year as index
# df_long.set_index('Year', inplace=True)
#
# # Convert index to datetime and set to year only
# df_long.index = pd.to_datetime(df_long.index, format='%Y').year

df_long

,Year,State,Population
0,2010,Alabama,4785298
1,2011,Alabama,4802740
2,2012,Alabama,4822023
3,2013,Alabama,4833722
4,2014,Alabama,4849377
5,2015,Alabama,4858979
6,2016,Alabama,4863300
7,2017,Alabama,4874747
8,2018,Alabama,4887871
9,2019,Alabama,4903185


In [15]:
# Empty DataFrame to store the forecasted results for each state
forecast_df = pd.DataFrame()

# Loop over each unique state in the DataFrame
for state in df_long['State'].unique():
    # Filter the DataFrame for the current state
    state_df = df_long[df_long['State'] == state]

    # Reset the index
    state_df = state_df.reset_index()

    # Make sure the index is a datetime index with frequency set to 'A'
    state_df.index = pd.DatetimeIndex(pd.to_datetime(state_df['Year'], format='%Y')).to_period('A')

    # Create and fit the ARIMA model
    model = ARIMA(state_df['Population'], order=(1, 1, 1))
    model_fit = model.fit()

    # Forecast the population for the next 10 years
    forecast = model_fit.predict(len(state_df), len(state_df) + 9)

    # Prepare forecasted data for merging with original data
    forecast_data = pd.DataFrame({'Year': forecast.index.year, 'State': state, 'Population': forecast.values})

    # Concatenate forecasted data to the main forecast DataFrame
    forecast_df = pd.concat([forecast_df, forecast_data], ignore_index=True)

# Convert 'Year' in forecast_df to integer
forecast_df['Year'] = forecast_df['Year'].astype(int)

# Concatenate the historical and forecast dataframes
population_complete = pd.concat([df_long, forecast_df])

# Sort by 'Year' and 'State'
population_complete = population_complete.sort_values(['Year', 'State'])

# Pivot the DataFrame to have 'State' as index, 'Year' as columns, and 'Population' as values
population_complete = population_complete.pivot(index='State', columns='Year', values='Population')

# Convert values to integers
population_complete = population_complete.astype(int)

population_complete

C:\Users\ChrisPeoples\anaconda3\envs\2023_EV_Infrastructure_Trends\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\ChrisPeoples\anaconda3\envs\2023_EV_Infrastructure_Trends\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
C:\Users\ChrisPeoples\anaconda3\envs\2023_EV_Infrastructure_Trends\lib\site-packages\statsmodels\tsa\statespace\sarimax.py:978: UserWarning: Non-invertible starting MA parameters found. Using zeros as starting parameters.
  warn('Non-invertible starting MA parameters found.'
C:\Users\ChrisPeoples\anaconda3\envs\2023_EV_Infrastructure_Trends\lib\site-packages\statsmodels\base\model.py:604: ConvergenceWarning: Ma

Year,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026,2027,2028,2029
State,,,,,,,,,,,,,,,,,,,,
Alabama,4785298,4802740,4822023,4833722,4849377,4858979,4863300,4874747,4887871,4903185,4912840,4921953,4930554,4938672,4946333,4953563,4960387,4966828,4972906,4978643
Alaska,713985,722718,731449,735132,736732,738432,741894,739795,737438,731545,731551,731554,731557,731558,731559,731560,731560,731560,731560,731560
Arizona,6413737,6482505,6553255,6626624,6731484,6828065,6931071,7016270,7171646,7278717,7374280,7469813,7565315,7660785,7756225,7851634,7947012,8042359,8137675,8232960
Arkansas,2921606,2937979,2949131,2959373,2966369,2978204,2988248,3004279,3013825,3017804,3025601,3032940,3039847,3046348,3052465,3058223,3063642,3068742,3073541,3078058
California,37349363,37691912,38041430,38332521,38802500,39144818,39250017,39536653,39557045,39512223,39616385,39705692,39782261,39847910,39904196,39952455,39993831,40029306,40059721,40085798
Colorado,5049071,5116796,5187582,5268367,5355866,5456574,5540545,5607154,5695564,5758736,5837000,5915154,5993199,6071135,6148961,6226679,6304288,6381787,6459179,6536462
Connecticut,3577073,3580709,3590347,3596080,3596677,3590886,3576452,3588184,3572665,3565287,3564432,3563600,3562787,3561995,3561223,3560470,3559735,3559019,3558321,3557639
Delaware,899769,907135,917092,925749,935614,945934,952065,961939,967171,973764,980665,987336,993782,1000011,1006032,1011850,1017473,1022908,1028159,1033235
District of Columbia,604453,617996,632323,646449,658893,672228,681170,693972,702455,705749,712892,719449,725468,730993,736065,740720,744994,748917,752518,755824


In [16]:
# Save dataframe using the save_data function
save_data(population_complete, 'USA_State_Population')

Saving data USA_State_Population.csv
File saved at C:\Users\ChrisPeoples\OneDrive - Peoples Partners and Associates, LLC\PP&A Working Directory\70_Studies_and_Reports\2023_EV_Infrastructure_Trends\data\USA_State_Population.csv/USA_State_Population.csv


#### EV Charging Infrastructure (NREL Data)

##### Build Dataframe structure for charging infrastructure data

In [17]:
# define the dataframe columns
column_names = ['ID',
                'Name',
                'EV Network',
                'Owner Type',
                'Pricing', 
                'Renewable Source',
                'Access Type',
                'Access Details',
                'Access Times',
                'Connector Types',
                'Level 1 Chargers Count',
                'Level 2 Chargers Count',
                'DC Fast Chargers Count',
                'Other Charger Count',
                'Street Address',
                'City',
                'State',
                'Zip Code', 
                'Latitude', 
                'Longitude'] 

# instantiate the dataframe
ev_charger_structure = pd.DataFrame(columns=column_names)
ev_charger_structure

,ID,Name,EV Network,Owner Type,Pricing,Renewable Source,Access Type,Access Details,Access Times,Connector Types,Level 1 Chargers Count,Level 2 Chargers Count,DC Fast Chargers Count,Other Charger Count,Street Address,City,State,Zip Code,Latitude,Longitude


##### Selected Data Definitions
<a href="https://developer.nrel.gov/docs/transportation/alt-fuel-stations-v1/get/">Link to NREL Documentation</a>
<section>
    <h5 style="text-decoration: underline; line-height: 0; margin-top: 20px;">Ownership Type:</h5>
<ul>
    <li>P - Privately Owned</li>
    <li>T - Utility Owned</li>
    <li>FG - Federal Government Owned</li>
    <li>LG - Local Government</li>
    <li>SG - State Government</li>
    <li>J - Joint Ownership</li>
</ul>
</section>

<section>
    <h5 style="text-decoration: underline; line-height: 0; margin-top: 20px;">EV Connector Types:</h5>
<ul>
    <li>NEMA515 - NEMA 5 - 15 (Level 1)</li>
    <li>NEMA520 - NEMA 5 - 20 (Level 1)</li>
    <li>NEMA1450 - NEMA 14 - 50 (Level 1)</li>
    <li>J1772 - J1772 (Level 2)</li>
    <li>CHADEMO - HAdeMO (DC Fast Charging)</li>
    <li>J1772COMBO - SAE J1772 Combo (DC Fast Charging)</li>
    <li>TESLA - Tesla (DC Fast Charging)</li>
</ul>
</section>

<section>
    <h5 style="text-decoration: underline; line-height: 0; margin-top: 20px;">Other Charger Types:</h5>
<ul>
    <li>SP Inductive - Small Paddle Inductive</li>
    <li>LP Inductive - Large Paddle Inductive</li>
    <li>Avcon - Conductive</li>
</ul>
</section>

<section>
    <h5 style="text-decoration: underline; line-height: 0; margin-top: 20px;">Access Details:</h5>
<ul>
    <li>Public - Publicly available to all customers</li>
    <li>Public - Call ahead - Publicly available, but customers should call before visiting</li>
    <li>Public - Credit card at all times - Publicly available, but only accepts credit cards as payment. The station may also except fleet cards or station-specific fueling cards</li>
    <li>Public - Card key at all times - Publicly available, but only accepts fleet cards or station-specific fueling cards</li>
    <li>Public - Credit card after hours - Publicly available and accepts credit cards 24 hours a day. The station may also except fleet cards or station-specific fueling cards</li>
    <li>Public - Card key after hours - Publicly available 24 hours a day, but only accepts fleet cards or station-specific fueling cards</li>
</ul>
</section>

##### Prepare NREL query statement

In [18]:
url = "https://developer.nrel.gov/api/alt-fuel-stations/v1.json?fuel_type=ELEC&api_key={}".format(NREL_ID)
NREL_ev_data = requests.get(url).json()
NREL_ev_data = NREL_ev_data['fuel_stations']

##### Function to convert NREL data into dataframe format

In [19]:
def get_EV_data(data_input):
    rows = []

    # Loop to convert JSON results to dataframe rows
    for ind, entry in enumerate(data_input):
        connector_types = entry['ev_connector_types'] if entry['ev_connector_types'] else []

        # Check for status code and assign the appropriate description
        status_code = entry['status_code']
        if status_code == 'E':
            status_description = 'Available'
        elif status_code == 'P':
            status_description = 'Planned'
        elif status_code == 'T':
            status_description = 'Temporarily Unavailable'
        else:
            status_description = 'Unknown'

        # Assign data entry data to a dictionary
        row = {
            'ID': entry['id'],
            'Status': status_description,
            'Open Date': entry['open_date'],
            'Name': entry['station_name'],
            'EV Network': entry['ev_network'],
            'Owner Type': entry['owner_type_code'],
            'Pricing': entry['ev_pricing'],
            'Renewable Source': entry['ev_renewable_source'],
            'Access Type': entry['access_code'],
            'Access Details': entry['groups_with_access_code'],
            'Access Times': entry['access_days_time'],
            'Level 1 Chargers Count': entry['ev_level1_evse_num'],
            'Level 2 Chargers Count': entry['ev_level2_evse_num'],
            'DC Fast Chargers Count': entry["ev_dc_fast_num"],
            'Other Charger Count': entry['ev_other_evse'],
            'NEMA 14-50': int('NEMA1450' in connector_types),
            'NEMA 5-15': int('NEMA515' in connector_types),
            'NEMA 5-20': int('NEMA520' in connector_types),
            'J1772': int('J1772' in connector_types),
            'CCS': int('J1772COMBO' in connector_types),
            'CHAdeMO': int('CHADEMO' in connector_types),
            'Tesla': int('TESLA' in connector_types),
            'Facility Type': entry['facility_type'],
            'Street Address': entry['street_address'],
            'City': entry['city'],
            'State': entry['state'],
            'Zip Code': entry['zip'],
            'Latitude': entry['latitude'],
            'Longitude': entry['longitude'],
            'Expected Date': entry['expected_date']
        }

        # Append the row dictionary to the list
        rows.append(row)

    # Create a DataFrame from the list of dictionaries
    output = pd.DataFrame(rows)

    # Output dataframe
    return output

print("Ready")

Ready


In [20]:
ev_data = get_EV_data(NREL_ev_data)
print("Done")

Done


In [21]:
print(ev_data.shape)
ev_data.head(20)

(61913, 30)


,ID,Status,Open Date,Name,EV Network,Owner Type,Pricing,Renewable Source,Access Type,Access Details,Access Times,Level 1 Chargers Count,Level 2 Chargers Count,DC Fast Chargers Count,Other Charger Count,NEMA 14-50,NEMA 5-15,NEMA 5-20,J1772,CCS,CHAdeMO,Tesla,Facility Type,Street Address,City,State,Zip Code,Latitude,Longitude,Expected Date
0,1517,Available,1999-10-15,LADWP - Truesdale Center,Non-Networked,LG,None,None,private,Private,Fleet use only,NaN,39.0,3.0,None,0,0,0,1,1,1,0,UTILITY,11797 Truesdale St,Sun Valley,CA,91352,34.248319,-118.387971,None
1,1519,Available,2020-02-28,LADWP - West LA District Office,Non-Networked,LG,Free,None,private,Private,None,NaN,4.0,NaN,None,0,0,0,1,0,0,0,UTILITY,1394 S Sepulveda Blvd,Los Angeles,CA,90024,34.052542,-118.448504,None
2,1523,Available,1995-08-30,Los Angeles Convention Center,Non-Networked,P,Free; parking fee,None,public,Public,5:30am-9pm; pay lot,NaN,7.0,NaN,None,0,0,0,1,0,0,0,PARKING_GARAGE,1201 S Figueroa St,Los Angeles,CA,90015,34.040539,-118.271387,None
3,1525,Available,1999-10-15,LADWP - John Ferraro Building,Non-Networked,LG,None,None,private,Private,For fleet and employee use only,NaN,311.0,2.0,None,0,0,0,1,1,1,0,UTILITY,111 N Hope St,Los Angeles,CA,90012,34.059133,-118.248589,None
4,1531,Available,2018-05-01,LADWP - Haynes Power Plant,Non-Networked,LG,None,None,private,Private,Fleet use only,NaN,19.0,1.0,None,0,0,0,1,1,1,0,UTILITY,6801 E 2nd St,Long Beach,CA,90803,33.759802,-118.096665,None
5,1552,Available,1999-10-15,LADWP - Harbor Generating Station,Non-Networked,LG,None,None,private,Private,Fleet use only,NaN,10.0,NaN,None,0,0,0,1,0,0,0,UTILITY,161 N Island Ave,Wilmington,CA,90744,33.770508,-118.265628,None
6,1556,Available,2016-01-01,LADWP - Sylmar West,Non-Networked,LG,None,None,private,Private - Government only,Fleet use only,NaN,2.0,NaN,None,0,0,0,1,0,0,0,UTILITY,13201 Sepulveda Blvd,Sylmar,CA,91342,34.303090,-118.480505,None
7,1572,Available,1999-10-15,LADWP - EV Service Center,Non-Networked,LG,None,None,private,Private,Fleet and employee use only,NaN,46.0,1.0,None,0,0,0,1,0,1,0,UTILITY,1630 N Main St,Los Angeles,CA,90012,34.066801,-118.227605,None
8,1573,Available,2019-04-01,LADWP - Fairfax Center,Non-Networked,LG,None,None,private,Private,Fleet use only,NaN,13.0,NaN,None,0,0,0,1,0,0,0,UTILITY,2311 S Fairfax Ave,Los Angeles,CA,90016,34.036777,-118.368841,None
9,1583,Available,1996-10-15,California Air Resources Board,Non-Networked,SG,Free,None,public,Public,24 hours daily,NaN,3.0,NaN,None,0,0,0,1,0,0,0,STATE_GOV,9530 Telstar Ave,El Monte,CA,91731,34.068720,-118.064000,None


In [22]:
ev_data.dtypes

ID                          int64
Status                     object
Open Date                  object
Name                       object
EV Network                 object
Owner Type                 object
Pricing                    object
Renewable Source           object
Access Type                object
Access Details             object
Access Times               object
Level 1 Chargers Count    float64
Level 2 Chargers Count    float64
DC Fast Chargers Count    float64
Other Charger Count        object
NEMA 14-50                  int64
NEMA 5-15                   int64
NEMA 5-20                   int64
J1772                       int64
CCS                         int64
CHAdeMO                     int64
Tesla                       int64
Facility Type              object
Street Address             object
City                       object
State                      object
Zip Code                   object
Latitude                  float64
Longitude                 float64
Expected Date 

##### Clean EV Infrastructure Data Set

Cleaning tasks:
*   Replace 'None' values in charger columns
*   Replace Pricing, Access Times, and Expected Dates 'None' values with not available indicators
*   Set variables to Int type



In [23]:
# Replace 'None' values in charger columns
chargers = ['Level 1 Chargers Count', 'Level 2 Chargers Count', 'DC Fast Chargers Count', 'Other Charger Count']
ev_data[chargers] = ev_data[chargers].replace(np.nan, 0)

# Replace Pricing, Access Times, and Expected Dates 'None' values with not available indicators
cols = ['Pricing', 'Access Times', 'Expected Date', 'Owner Type', 'Renewable Source']
ev_data[cols] = ev_data[cols].replace(np.nan, 'NA')

# Set variables to Int type
# ev_data['Zip Code'] = ev_data['Zip Code'].astype(int)
ev_data['Level 1 Chargers Count'] = ev_data['Level 1 Chargers Count'].astype(int)
ev_data['Level 2 Chargers Count'] = ev_data['Level 2 Chargers Count'].astype(int)
ev_data['DC Fast Chargers Count'] = ev_data['DC Fast Chargers Count'].astype(int)
print("Clean")

Clean


##### Validate NREL Data Completeness

In [24]:
missing_data = ev_data.isnull()

for column in missing_data.columns.values.tolist():
    print(column)
    print (missing_data[column].value_counts())
    print("") 

ID
ID
False    61913
Name: count, dtype: int64

Status
Status
False    61913
Name: count, dtype: int64

Open Date
Open Date
False    61803
True       110
Name: count, dtype: int64

Name
Name
False    61913
Name: count, dtype: int64

EV Network
EV Network
False    61909
True         4
Name: count, dtype: int64

Owner Type
Owner Type
False    61913
Name: count, dtype: int64

Pricing
Pricing
False    61913
Name: count, dtype: int64

Renewable Source
Renewable Source
False    61913
Name: count, dtype: int64

Access Type
Access Type
False    61912
True         1
Name: count, dtype: int64

Access Details
Access Details
False    61913
Name: count, dtype: int64

Access Times
Access Times
False    61913
Name: count, dtype: int64

Level 1 Chargers Count
Level 1 Chargers Count
False    61913
Name: count, dtype: int64

Level 2 Chargers Count
Level 2 Chargers Count
False    61913
Name: count, dtype: int64

DC Fast Chargers Count
DC Fast Chargers Count
False    61913
Name: count, dtype: int64

Other

In [25]:
ev_data.dtypes

ID                          int64
Status                     object
Open Date                  object
Name                       object
EV Network                 object
Owner Type                 object
Pricing                    object
Renewable Source           object
Access Type                object
Access Details             object
Access Times               object
Level 1 Chargers Count      int32
Level 2 Chargers Count      int32
DC Fast Chargers Count      int32
Other Charger Count        object
NEMA 14-50                  int64
NEMA 5-15                   int64
NEMA 5-20                   int64
J1772                       int64
CCS                         int64
CHAdeMO                     int64
Tesla                       int64
Facility Type              object
Street Address             object
City                       object
State                      object
Zip Code                   object
Latitude                  float64
Longitude                 float64
Expected Date 

In [26]:
ev_data.head()

,ID,Status,Open Date,Name,EV Network,Owner Type,Pricing,Renewable Source,Access Type,Access Details,Access Times,Level 1 Chargers Count,Level 2 Chargers Count,DC Fast Chargers Count,Other Charger Count,NEMA 14-50,NEMA 5-15,NEMA 5-20,J1772,CCS,CHAdeMO,Tesla,Facility Type,Street Address,City,State,Zip Code,Latitude,Longitude,Expected Date
0,1517,Available,1999-10-15,LADWP - Truesdale Center,Non-Networked,LG,NA,NA,private,Private,Fleet use only,0,39,3,0,0,0,0,1,1,1,0,UTILITY,11797 Truesdale St,Sun Valley,CA,91352,34.248319,-118.387971,NA
1,1519,Available,2020-02-28,LADWP - West LA District Office,Non-Networked,LG,Free,NA,private,Private,NA,0,4,0,0,0,0,0,1,0,0,0,UTILITY,1394 S Sepulveda Blvd,Los Angeles,CA,90024,34.052542,-118.448504,NA
2,1523,Available,1995-08-30,Los Angeles Convention Center,Non-Networked,P,Free; parking fee,NA,public,Public,5:30am-9pm; pay lot,0,7,0,0,0,0,0,1,0,0,0,PARKING_GARAGE,1201 S Figueroa St,Los Angeles,CA,90015,34.040539,-118.271387,NA
3,1525,Available,1999-10-15,LADWP - John Ferraro Building,Non-Networked,LG,NA,NA,private,Private,For fleet and employee use only,0,311,2,0,0,0,0,1,1,1,0,UTILITY,111 N Hope St,Los Angeles,CA,90012,34.059133,-118.248589,NA
4,1531,Available,2018-05-01,LADWP - Haynes Power Plant,Non-Networked,LG,NA,NA,private,Private,Fleet use only,0,19,1,0,0,0,0,1,1,1,0,UTILITY,6801 E 2nd St,Long Beach,CA,90803,33.759802,-118.096665,NA


##### Filter to Baltimore EV Infrastructure Data

In [27]:
ev_baltimore = ev_data[ev_data["City"]=='Baltimore']
print(ev_baltimore.shape)
ev_baltimore.head()

(313, 30)


,ID,Status,Open Date,Name,EV Network,Owner Type,Pricing,Renewable Source,Access Type,Access Details,Access Times,Level 1 Chargers Count,Level 2 Chargers Count,DC Fast Chargers Count,Other Charger Count,NEMA 14-50,NEMA 5-15,NEMA 5-20,J1772,CCS,CHAdeMO,Tesla,Facility Type,Street Address,City,State,Zip Code,Latitude,Longitude,Expected Date
474,40012,Available,2011-08-15,Municipal Garage,Non-Networked,LG,Free,NA,public,Public,24 hours daily; Drivers must bring their own J...,1,2,0,0,0,0,1,1,0,0,0,FLEET_GARAGE,200 N Holliday St,Baltimore,MD,21202,39.291785,-76.610539,NA
653,43190,Available,2012-01-15,UM PTS GRAND,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,5 N Paca St,Baltimore,MD,21201,39.290036,-76.621894,NA
654,43191,Available,2012-01-15,UM PTS PEARL #3,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,622 W Fayette St,Baltimore,MD,21201,39.290764,-76.624390,NA
655,43195,Available,2012-01-15,UM PTS PRATT,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,646 West Pratt Street,Baltimore,MD,21201,39.286510,-76.624935,NA
656,43196,Available,2012-01-15,UM PTS PLAZA #1,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,500 West Redwood Street,Baltimore,MD,21201,39.288080,-76.622679,NA


### Fuel Prices | Northeastern Region (incl. Maryland)

<div>Gas price data is unavailable via API at present but can be found at the following link:</div>
<div><a href="https://www.eia.gov/dnav/pet/hist/LeafHandler.ashx?n=PET&s=EMM_EPMR_PTE_R1Y_DPG&f=M">Gas Price Data</a></div>

In [28]:
# Pull the data
url = "https://api.eia.gov/v2/petroleum/pri/gnd/data/?frequency=monthly&data[0]=value&facets[series][]=EMM_EPMR_PTE_R1Y_DPG&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key={}".format(NREL_ID)

response = requests.get(url)

# The data is returned as JSON
data = response.json()

print(data)
#
# # Assume the data is in the first series (you should adjust this according to the actual structure of the response)
# series_data = data['series'][0]['data']
#
# # Convert the data to a DataFrame
# df = pd.DataFrame(series_data, columns=['Date', 'Value'])
#
# # Print the DataFrame
# print(df)

{'response': {'total': 726, 'dateFormat': 'YYYY-MM', 'frequency': 'monthly', 'data': [{'period': '2023-06', 'duoarea': 'R1Y', 'area-name': 'PADD 1B', 'product': 'EPMR', 'product-name': 'Regular Gasoline', 'process': 'PTE', 'process-name': 'Retail Sales', 'series': 'EMM_EPMR_PTE_R1Y_DPG', 'series-description': 'Central Atlantic (PADD 1B) Regular All Formulations Retail Gasoline Prices (Dollars per Gallon)', 'value': 3.572, 'units': '$/GAL'}, {'period': '2023-05', 'duoarea': 'R1Y', 'area-name': 'PADD 1B', 'product': 'EPMR', 'product-name': 'Regular Gasoline', 'process': 'PTE', 'process-name': 'Retail Sales', 'series': 'EMM_EPMR_PTE_R1Y_DPG', 'series-description': 'Central Atlantic (PADD 1B) Regular All Formulations Retail Gasoline Prices (Dollars per Gallon)', 'value': 3.573, 'units': '$/GAL'}, {'period': '2023-04', 'duoarea': 'R1Y', 'area-name': 'PADD 1B', 'product': 'EPMR', 'product-name': 'Regular Gasoline', 'process': 'PTE', 'process-name': 'Retail Sales', 'series': 'EMM_EPMR_PTE_R1Y

In [29]:
# Convert to dataframes

# Extract the data points
series_data = data['response']['data']

# Convert the data to a DataFrame
df = pd.DataFrame(series_data)
df['period'] = pd.to_datetime(df['period'])  # Convert 'period' column to datetime

df.head()

# Extract the gas prices
gas_prices = df[['period', 'value']]

# Rename the columns
gas_prices.columns = ['date', 'gas_price']

# Filter to specific date range
gas_prices = gas_prices[(gas_prices['date'] >= '2019-01-01') & (gas_prices['date'] <= '2023-12-01')]
gas_prices

,date,gas_price
0,2023-06-01,3.572
1,2023-05-01,3.573
2,2023-04-01,3.554
3,2023-03-01,3.406
4,2023-02-01,3.458
5,2023-01-01,3.469
6,2022-12-01,3.472
7,2022-11-01,3.851
8,2022-10-01,3.702
9,2022-09-01,3.710


### Electricity Prices | Maryland Electricity Price Data

In [30]:
# Pull the data
url = "https://api.eia.gov/v2/electricity/retail-sales/data/?frequency=monthly&data[0]=price&facets[stateid][]=MD&start=2019-01&sort[0][column]=period&sort[0][direction]=desc&offset=0&length=5000&api_key={}".format(NREL_ID)

response = requests.get(url)

# The data is returned as JSON
data = response.json()

print(data)

{'response': {'total': 312, 'dateFormat': 'YYYY-MM', 'frequency': 'monthly', 'data': [{'period': '2023-04', 'stateid': 'MD', 'stateDescription': 'Maryland', 'sectorid': 'IND', 'sectorName': 'industrial', 'price': 9.61, 'price-units': 'cents per kilowatthour'}, {'period': '2023-04', 'stateid': 'MD', 'stateDescription': 'Maryland', 'sectorid': 'OTH', 'sectorName': 'other', 'price': None, 'price-units': 'cents per kilowatthour'}, {'period': '2023-04', 'stateid': 'MD', 'stateDescription': 'Maryland', 'sectorid': 'RES', 'sectorName': 'residential', 'price': 15.92, 'price-units': 'cents per kilowatthour'}, {'period': '2023-04', 'stateid': 'MD', 'stateDescription': 'Maryland', 'sectorid': 'TRA', 'sectorName': 'transportation', 'price': 11.18, 'price-units': 'cents per kilowatthour'}, {'period': '2023-04', 'stateid': 'MD', 'stateDescription': 'Maryland', 'sectorid': 'ALL', 'sectorName': 'all sectors', 'price': 13.87, 'price-units': 'cents per kilowatthour'}, {'period': '2023-04', 'stateid': 'M

In [31]:
# Convert to dataframes
# Convert to dataframes

# Extract the data points
series_data = data['response']['data']

# Convert the data to a DataFrame
df = pd.DataFrame(series_data)
df['period'] = pd.to_datetime(df['period'])  # Convert 'period' column to datetime

# Filter to sectorName colume to residential
df = df[df['sectorName'] == 'residential']

# Extract electricity prices
electricity_prices = df[['period', 'price']]

# Rename the columns
electricity_prices.columns = ['date', 'resi_electricity_price']

# Filter to specific date range
electricity_prices = electricity_prices[(electricity_prices['date'] >= '2019-01-01') & (electricity_prices['date'] <= '2023-12-01')]

# Index by date
electricity_prices = electricity_prices.set_index('date')

electricity_prices

,resi_electricity_price
date,
2023-04-01,15.92
2023-03-01,15.76
2023-02-01,16.12
2023-01-01,15.87
2022-12-01,15.56
2022-11-01,15.42
2022-10-01,15.89
2022-09-01,14.61
2022-08-01,14.30


### Geospatial Data - Maps of Counties and Neigbborhoods

#### Gather Maryland County Coordinates and Locational Data

In [32]:
import requests

url = 'https://raw.githubusercontent.com/frankrowe/maryland-geojson/master/maryland-counties.geojson'
response = requests.get(url)

with open('maryland-counties.geojson', 'w') as file:
    file.write(response.text)

with open('maryland-counties.geojson', 'r') as file:
    maryland_counties = file.read()

counties = json.loads(maryland_counties)
counties = counties['features']
counties[0]

{'type': 'Feature',
 'properties': {'name': 'Allegany'},
 'geometry': {'type': 'Polygon',
  'coordinates': [[[-78.72247849700874, 39.72293680259466],
    [-78.34273160144357, 39.72245780651755],
    [-78.34603570538482, 39.72132852481378],
    [-78.34078320859582, 39.716239266427365],
    [-78.33828435426172, 39.716772506811225],
    [-78.3362139385028, 39.71990977860767],
    [-78.33356195554411, 39.71899928707819],
    [-78.33183831010778, 39.71446936687261],
    [-78.329167802137, 39.712280300846366],
    [-78.33420965187652, 39.71257069623337],
    [-78.33434352854464, 39.70990131981943],
    [-78.31836983766253, 39.702583041971636],
    [-78.3173195949011, 39.69930848103388],
    [-78.31985331880286, 39.69574118937848],
    [-78.31333257364668, 39.69239895921175],
    [-78.31663150625955, 39.68910520182264],
    [-78.32280614451973, 39.68900721927672],
    [-78.32202451617927, 39.68736973541037],
    [-78.314922794248, 39.685563439381646],
    [-78.3166051204736, 39.68316600240325

#### Gather Baltimore Neighborhood Coordinates and Locational Data

<a href="https://github.com/blackmad/neighborhoods">Link to Neighborhood JSON</a>

In [33]:
url = 'https://raw.githubusercontent.com/codeforgermany/click_that_hood/main/public/data/baltimore.geojson'
response = requests.get(url)

with open('baltimore.geojson', 'w') as file:
    file.write(response.text)

with open('baltimore.geojson', 'r') as file:
    baltimore_neighborhoods = file.read()

baltimore_neighborhoods = json.loads(baltimore_neighborhoods)
baltimore_neighborhoods = baltimore_neighborhoods['features']
baltimore_neighborhoods[0]

{'type': 'Feature',
 'properties': {'name': 'Armistead Gardens',
  'cartodb_id': 5,
  'created_at': '2013-11-12T15:45:44Z',
  'updated_at': '2013-11-12T15:45:44Z'},
 'geometry': {'type': 'MultiPolygon',
  'coordinates': [[[[-76.5588, 39.306457],
     [-76.55892, 39.306536],
     [-76.559181, 39.306711],
     [-76.559334, 39.306809],
     [-76.559387, 39.306847],
     [-76.55994, 39.307247],
     [-76.560362, 39.307539],
     [-76.56074, 39.307814],
     [-76.561224, 39.308173],
     [-76.56136, 39.308271],
     [-76.560687, 39.308707],
     [-76.560426, 39.308846],
     [-76.559794, 39.309183],
     [-76.55842, 39.309915],
     [-76.558332, 39.309966],
     [-76.558005, 39.310157],
     [-76.557638, 39.310351],
     [-76.557575, 39.31038],
     [-76.555834, 39.311363],
     [-76.555223, 39.311682],
     [-76.555006, 39.311834],
     [-76.5546, 39.312104],
     [-76.550387, 39.314907],
     [-76.549787, 39.315307],
     [-76.549164, 39.315691],
     [-76.549164, 39.31567],
     [-76.549

In [34]:
baltimore_neighborhoods

[{'type': 'Feature',
  'properties': {'name': 'Armistead Gardens',
   'cartodb_id': 5,
   'created_at': '2013-11-12T15:45:44Z',
   'updated_at': '2013-11-12T15:45:44Z'},
  'geometry': {'type': 'MultiPolygon',
   'coordinates': [[[[-76.5588, 39.306457],
      [-76.55892, 39.306536],
      [-76.559181, 39.306711],
      [-76.559334, 39.306809],
      [-76.559387, 39.306847],
      [-76.55994, 39.307247],
      [-76.560362, 39.307539],
      [-76.56074, 39.307814],
      [-76.561224, 39.308173],
      [-76.56136, 39.308271],
      [-76.560687, 39.308707],
      [-76.560426, 39.308846],
      [-76.559794, 39.309183],
      [-76.55842, 39.309915],
      [-76.558332, 39.309966],
      [-76.558005, 39.310157],
      [-76.557638, 39.310351],
      [-76.557575, 39.31038],
      [-76.555834, 39.311363],
      [-76.555223, 39.311682],
      [-76.555006, 39.311834],
      [-76.5546, 39.312104],
      [-76.550387, 39.314907],
      [-76.549787, 39.315307],
      [-76.549164, 39.315691],
      [-76.

##### Function to gather neighborhood coordinates

In [35]:
# Function to gather neighborhood coordinates
def gather_coords(neighborhood_item):

    list_t = []

    # Iterate through neighborhood coordinates
    for coord in neighborhood_item:

        contains = (coord[0], coord[1])
        list_t.append(contains)

    return list_t
print("Ready")

Ready


##### Build neighborhood dictionary used for validating charger neighborhood location

In [36]:
# Build a dictionary from the Baltimore City GeoJSON consisting of neighborhood names and it's respective coordinates

# Set up target variables
neighborhood_array = []
neighborhood_dict = {'Name':'', 'Coords':[]}

# Initialize GeoJSON data target
# neighborhood_items = city_input['features']

# Iterate through JSON to pull neighborhood details
for items in baltimore_neighborhoods:
    
    # Capture name
    name = items['properties']['name']
    
    # Capture coords 
    coords = gather_coords(items['geometry']['coordinates'][0][0])
    
    # Update dictionary
    neighborhood_dict.update({'Name':name, 'Coords': coords})
    
    # Add to output list
    neighborhood_array.append(dict(neighborhood_dict))
print("Ready")

Ready


##### Generate neighborhood list with coordinates for use with Venues

In [37]:
from geographiclib.geodesic import Geodesic
from geographiclib.polygonarea import PolygonArea

geod = Geodesic.WGS84

def calculate_area(polygon_data):
    poly_area = PolygonArea(geod)
    for point in polygon_data:
        poly_area.AddPoint(point[1], point[0])

    _, _, area = poly_area.Compute()

    # If the area is negative, reverse the order of the coordinates and recompute the area
    if area < 0:
        polygon_data = polygon_data[::-1]
        poly_area = PolygonArea(geod)
        for point in polygon_data:
            poly_area.AddPoint(point[1], point[0])
        _, _, area = poly_area.Compute()

    return area

In [38]:
from shapely.geometry import Polygon

# Loop to find centroid
for neighborhood in neighborhood_array:
    
    # Assign neighborhood coordinates to polygon_data variable
    polygon_data = neighborhood['Coords']
    
    # Calculate centroid via Shapely
    ref_polygon = Polygon(polygon_data)
    
    # Assign center lat and lon points to neighborhood array
    neighborhood['Center Longitude'] = ref_polygon.centroid.x
    neighborhood['Center Latitude'] = ref_polygon.centroid.y
    neighborhood['Area'] = calculate_area(polygon_data) * 3.86102e-7 # Area is in Square Miles convert by using the constant 3.86102e-7

##### Build neighborhood dataframe from neighborhood array

In [39]:
# Initialize list variables
neighborhood_list = []
center_long = []
center_lat = []
area = []

# Loop to Build Neighborhood list
for items in neighborhood_array:
    neighborhood_list.append(items['Name'])
    center_long.append(items['Center Longitude'])
    center_lat.append(items['Center Latitude'])
    area.append(items['Area'])
    
# Convert Neighborhood list to dataframe for function loop
neighborhood_list = pd.DataFrame({"Neighborhood":neighborhood_list, 'Center Latitude':center_lat, 'Center Longitude':center_long, 'Area':area})
neighborhood_list

,Neighborhood,Center Latitude,Center Longitude,Area
0,Armistead Gardens,39.307342,-76.551451,0.472803
1,Wyman Park,39.331961,-76.626138,0.110365
2,Canton,39.282043,-76.575227,0.570498
3,Canton Industrial Area,39.268608,-76.555621,2.488602
4,Fairfield Area,39.238180,-76.581335,2.404451
5,Fells Point,39.283240,-76.593627,0.288077
6,Frankford,39.329278,-76.544633,2.124897
7,Franklintown,39.305836,-76.701453,0.595389
8,Garwyn Oaks,39.317854,-76.679329,0.131365
9,Heritage Crossing,39.296259,-76.628428,0.085181


### Supplement EV Charger Data with Neighborhood Data

#### Function to check whether or not a charging station falls within the boundaries of a neighborhood


In [40]:
def check_coords(long_t, lat_t, coords):
#     print('Made it here')
    point = Point(long_t, lat_t)
    polygon = Polygon(coords)

    return polygon.contains(point)
print("Ready")

Ready


#### Function to add neighborhood assignments to EV charging station data

In [41]:
def define_neighborhoods(dataframe):
    # Create an explicit copy of the input DataFrame
    dataframe = dataframe.copy()

    # Loop to generate a unique unit ID
    for i, j in dataframe.iterrows():

        # Capture latitude and longitude values from dataframe row
        longitude = j["Longitude"]
        latitude = j["Latitude"]

        # Iterate through neighborhood coordinates to check whether the coordinates are within a neighborhood
        for neighborhood in neighborhood_array:
            coords = neighborhood['Coords']
            name = neighborhood['Name']
            if check_coords(longitude, latitude, coords):
                dataframe.at[i, 'Neighborhood'] = name
                break
                
    # Output updated dataframe with Unit IDs
    return dataframe
print("Ready")

Ready


In [42]:
ev_data_baltimore = define_neighborhoods(ev_baltimore)
ev_data_baltimore.head()

,ID,Status,Open Date,Name,EV Network,Owner Type,Pricing,Renewable Source,Access Type,Access Details,Access Times,Level 1 Chargers Count,Level 2 Chargers Count,DC Fast Chargers Count,Other Charger Count,NEMA 14-50,NEMA 5-15,NEMA 5-20,J1772,CCS,CHAdeMO,Tesla,Facility Type,Street Address,City,State,Zip Code,Latitude,Longitude,Expected Date,Neighborhood
474,40012,Available,2011-08-15,Municipal Garage,Non-Networked,LG,Free,NA,public,Public,24 hours daily; Drivers must bring their own J...,1,2,0,0,0,0,1,1,0,0,0,FLEET_GARAGE,200 N Holliday St,Baltimore,MD,21202,39.291785,-76.610539,NA,Downtown
653,43190,Available,2012-01-15,UM PTS GRAND,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,5 N Paca St,Baltimore,MD,21201,39.290036,-76.621894,NA,Downtown
654,43191,Available,2012-01-15,UM PTS PEARL #3,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,622 W Fayette St,Baltimore,MD,21201,39.290764,-76.624390,NA,University Of Maryland
655,43195,Available,2012-01-15,UM PTS PRATT,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,646 West Pratt Street,Baltimore,MD,21201,39.286510,-76.624935,NA,University Of Maryland
656,43196,Available,2012-01-15,UM PTS PLAZA #1,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,500 West Redwood Street,Baltimore,MD,21201,39.288080,-76.622679,NA,University Of Maryland


#### Drop Non-Batlimore Chargers

In [43]:
print(ev_data_baltimore.shape)

# Create a new DataFrame without rows containing NaN in the "Neighborhood" column
ev_data_baltimore = ev_data_baltimore.dropna(subset=["Neighborhood"], axis=0)

# Reset index, because we dropped rows
ev_data_baltimore.reset_index(drop=True, inplace=True)

print(ev_data_baltimore.shape)
ev_data_baltimore

(313, 31)
(269, 31)


,ID,Status,Open Date,Name,EV Network,Owner Type,Pricing,Renewable Source,Access Type,Access Details,Access Times,Level 1 Chargers Count,Level 2 Chargers Count,DC Fast Chargers Count,Other Charger Count,NEMA 14-50,NEMA 5-15,NEMA 5-20,J1772,CCS,CHAdeMO,Tesla,Facility Type,Street Address,City,State,Zip Code,Latitude,Longitude,Expected Date,Neighborhood
0,40012,Available,2011-08-15,Municipal Garage,Non-Networked,LG,Free,NA,public,Public,24 hours daily; Drivers must bring their own J...,1,2,0,0,0,0,1,1,0,0,0,FLEET_GARAGE,200 N Holliday St,Baltimore,MD,21202,39.291785,-76.610539,NA,Downtown
1,43190,Available,2012-01-15,UM PTS GRAND,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,5 N Paca St,Baltimore,MD,21201,39.290036,-76.621894,NA,Downtown
2,43191,Available,2012-01-15,UM PTS PEARL #3,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,622 W Fayette St,Baltimore,MD,21201,39.290764,-76.624390,NA,University Of Maryland
3,43195,Available,2012-01-15,UM PTS PRATT,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,646 West Pratt Street,Baltimore,MD,21201,39.286510,-76.624935,NA,University Of Maryland
4,43196,Available,2012-01-15,UM PTS PLAZA #1,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,500 West Redwood Street,Baltimore,MD,21201,39.288080,-76.622679,NA,University Of Maryland
5,43440,Available,2012-01-31,EVSP WALGREENS STORE 5623,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,1,0,0,0,0,0,1,0,0,0,None,3801 Liberty Heights Ave,Baltimore,MD,21215,39.326834,-76.681801,NA,Forest Park
6,48038,Available,2012-05-31,MTA LTR MT WASHNGTN,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,5701 Smith Ave,Baltimore,MD,21209,39.368244,-76.652339,NA,Mount Washington
7,63262,Available,2014-09-12,OAK CREST TOWN CENTER,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,3700 E Northern Pkwy,Baltimore,MD,21206,39.358460,-76.536562,NA,Overlea
8,63824,Available,2014-10-21,UM PTS PLAZA #2,ChargePoint Network,NA,NA,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,None,500-598 W Redwood St,Baltimore,MD,21201,39.288696,-76.623385,NA,University Of Maryland
9,66639,Available,2015-01-12,Maryland Institute College of Art - Commons Hall,Non-Networked,P,Free,NA,public,Public,24 hours daily,0,2,0,0,0,0,0,1,0,0,0,COLLEGE_CAMPUS,130 McMechen St,Baltimore,MD,21217,39.308846,-76.625419,NA,Bolton Hill


#### Create a Total Chargers column in the EV charging station data

In [44]:
ev_data_baltimore['Total Chargers'] = ev_data_baltimore.loc[:,chargers].sum(axis=1)
ev_data_baltimore['Total Chargers'].sum()

759

#### Split EV Charging Data into Public and Private Charging Station Dataframes

In [45]:
public_chargers_baltimore = ev_data_baltimore[ev_data_baltimore['Access Type']=='public'].groupby(['Neighborhood','Access Type'])[['Total Chargers']].sum()
public_chargers_baltimore.reset_index(drop=False, inplace=True)
public_chargers_baltimore.rename(columns = {'Total Chargers':'Total Public Chargers'}, inplace = True)

# Use ev_data_baltimore instead of ev_data
private_chargers_baltimore = ev_data_baltimore[ev_data_baltimore['Access Type']=='private'].groupby(['Neighborhood','Access Type'])[['Total Chargers']].sum()
private_chargers_baltimore.reset_index(drop=False, inplace=True)
private_chargers_baltimore.rename(columns = {'Total Chargers':'Total Private Chargers'}, inplace = True)
print("Total Private Chargers: {} in Baltimore".format(private_chargers_baltimore['Total Private Chargers'].sum()))
print("Total Public Chargers: {} in Baltimore".format(public_chargers_baltimore['Total Public Chargers'].sum()))

Total Private Chargers: 64 in Baltimore
Total Public Chargers: 695 in Baltimore


### Vehicle Registrations | USA | All Vehicles by Type

#### Use BeautifulSoup to pull data from the US Registrations by type website
<a href="https://afdc.energy.gov/vehicle-registration">Source</a>

In [46]:
# Specify the URL
url = 'https://afdc.energy.gov/vehicle-registration'

# Send a GET request
response = requests.get(url)

# Parse the HTML content of the page with BeautifulSoup
soup = BeautifulSoup(response.content, 'html.parser')

# Find the table on the page
table = soup.find('table')

# Get the headers of the table but skip the first header because it's blank
headers = [header.text.strip() for header in table.find_all('th')][1:]

# Get the rows of the table
rows = table.find_all('tr')

# Get the data from each row
data = []
for row in rows[1:]:  # Skip the first row because it's the headers
    cols = row.find_all('td')
    cols = [col.text.strip() for col in cols]
    data.append(cols)

# Create a pandas DataFrame from the data
us_vehicle_registrations = pd.DataFrame(data, columns=headers)

# Drop the first row
us_vehicle_registrations = us_vehicle_registrations.drop(0)

# Remove commas from the all columns except for the first column
us_vehicle_registrations.iloc[:, 1:] = us_vehicle_registrations.iloc[:, 1:].replace(',', '', regex=True)

# Set all columns except for the first column to integer type
us_vehicle_registrations.iloc[:, 1:] = us_vehicle_registrations.iloc[:, 1:].astype(int)

us_vehicle_registrations.head()

,State,Electric (EV),Plug-In Hybrid Electric (PHEV),Hybrid Electric (HEV),Biodiesel,Ethanol/Flex (E85),Compressed Natural Gas (CNG),Propane,Hydrogen,Methanol,Gasoline,Diesel
1,Alabama,4700,3300,42500,40500,449500,500,100,0,0,4051000,123500
2,Alaska,1300,500,7300,7600,50100,100,0,0,0,464200,31700
3,Arizona,40700,15500,132200,51000,460400,900,900,0,0,5395300,191800
4,Arkansas,2400,1800,26100,28700,290200,300,0,0,0,2241600,88800
5,California,563100,315300,1355900,163600,1343200,12600,1500,11800,0,30512600,710500


#### Save the Total USA Vehicle Registrations data to an Excel file

In [47]:
# Save dataframe using the save_data function
save_data(us_vehicle_registrations, 'Total_USA_Vehicle_Registration_by_State')

Saving data Total_USA_Vehicle_Registration_by_State.csv
File saved at C:\Users\ChrisPeoples\OneDrive - Peoples Partners and Associates, LLC\PP&A Working Directory\70_Studies_and_Reports\2023_EV_Infrastructure_Trends\data\Total_USA_Vehicle_Registration_by_State.csv/Total_USA_Vehicle_Registration_by_State.csv


### Vehicle Registrations | USA By State | Electric Vehicles
At present, no U.S.-level API data source exists. As such, the data is pulled from the Department of Energy: Alternative Fuels Data Center website: <a href="https://afdc.energy.gov/data/10962">US EV Registrations</a>
After downloading the data, it is uploaded into the notebook from an Excel file. Name the file "US_EV_Registration_Data.xlsx" and save it in the data folder the main project directory.

In [48]:
# Define path to Excel file using output_directory variable
excel_file = os.path.join(output_directory, 'US_EV_Registration_Data.xlsx')

# Read in EV registration data from Excel file. the header is in row 3, and the data is in columns 2 and 3
us_ev_registrations = pd.read_excel(excel_file, sheet_name='EV Registration Counts', header=2, usecols="B:C")

# Set Registration Count column to integer type
us_ev_registrations['Registration Count'] = us_ev_registrations['Registration Count'].astype(int)

us_ev_registrations.head()

,State,Registration Count
0,Alabama,4750
1,Alaska,1290
2,Arizona,40740
3,Arkansas,2390
4,California,563070


### Vehicle Registrations | Maryland by County | All Vehicles (Annual)
<div>Total number of vehicles with active MD registrations by county as of each month end from July 2020 to most recent</div>
<div>Data is pulled from <span><a href="https://opendata.maryland.gov/Transportation/MVA-VEHICLE-REGISTRATION-by-COUNTY-from-2010-to-20/kqkd-4fx8">Maryland Open Data</a></span></div>

In [49]:
# Unauthenticated client only works with public data sets. Note 'None'
# in place of application token, and no username or password:
client = Socrata("opendata.maryland.gov", None)

# Initialize variables
results = []
limit = 2000
offset = 0

# Loop through the dataset using the offset parameter
while True:
    batch = client.get("kqkd-4fx8", limit=limit, offset=offset)
    if not batch:
        break
    results.extend(batch)
    offset += limit

# Convert to pandas DataFrame
total_reg_zip = pd.DataFrame.from_records(results)
total_reg_zip.head()

,fiscal_year,allegany,anne_arundel,baltimore,baltimore_city,calvert,caroline,carroll,cecil,charles,dorchester,frederick,garrett,harford,howard,kent,montgomery,prince_george_s,queen_anne_s,somerset,st_mary_s,talbot,washington,wicomico,worcester
0,FY 2010,62813,513278,660553,280793,91108,36699,176842,94305,138672,31742,226529,33303,235366,251713,21453,754641,626009,53779,20570,104488,42116,136894,86553,57117
1,FY 2011,62512,533106,663514,285394,91768,36667,177125,94563,140423,31672,227672,33202,237096,255497,21397,752503,633920,53938,20382,105942,42380,137204,88153,56627
2,FY 2012,62574,537670,669052,289229,93222,36728,177603,95254,141965,32416,230727,33459,236024,258164,21651,755353,643710,54159,20021,108276,42344,137716,87294,57767
3,FY 2013,61390,538501,669201,297990,92699,35749,176281,93408,141107,32056,228442,32942,234421,258498,21191,758413,640226,53259,19495,107802,41680,135787,86735,56148
4,FY 2014,61481,542768,680074,303542,93322,36019,178086,94146,143729,32553,231390,33231,236667,261320,21397,763346,653111,53842,20125,108794,42110,137007,87766,56655


In [50]:
# Save dataframe using the save_data function
save_data(total_reg_zip, 'MD_Total_Vehicle_Registration')

Saving data MD_Total_Vehicle_Registration.csv
File saved at C:\Users\ChrisPeoples\OneDrive - Peoples Partners and Associates, LLC\PP&A Working Directory\70_Studies_and_Reports\2023_EV_Infrastructure_Trends\data\MD_Total_Vehicle_Registration.csv/MD_Total_Vehicle_Registration.csv


### Vehicle Registrations | Maryland by Zip Code | Electric Vehicles (Monthly)

<div>Total number of electric and plug-in hybrid vehicles with active MD registrations by zip code as of each month end from July 2020 to most recent</div>
<div>Data is pulled from <span><a href="https://opendata.maryland.gov/Transportation/MD-MDOT-MVA-Electric-and-Plug-in-Hybrid-Vehicle-Re/tugr-unu9">Maryland Open Data</a></span></div>

In [51]:
# Unauthenticated client only works with public data sets. Note 'None'
# in place of application token, and no username or password:
client = Socrata("opendata.maryland.gov", None)

# Initialize variables
results = []
limit = 2000
offset = 0

# Loop through the dataset using the offset parameter
while True:
    batch = client.get("tugr-unu9", limit=limit, offset=offset)
    if not batch:
        break
    results.extend(batch)
    offset += limit

# Convert to pandas DataFrame
ev_reg_zip = pd.DataFrame.from_records(results)
ev_reg_zip.head()

,year_month,fuel_category,zip_code,count
0,2020/07,Electric,19973,1
1,2020/07,Electric,20601,21
2,2020/07,Electric,20602,26
3,2020/07,Electric,20603,54
4,2020/07,Electric,20607,35


### Vehicle Registrations | Maryland by County | Electric Vehicles (Monthly)
<div>Total number of electric and plug-in hybrid vehicles with active MD registrations by county as of each month end from July 2020 to February 2023</div>
<div>Data is pulled from <span><a href="https://opendata.maryland.gov/Transportation/MDOT-MVA-Electric-and-Plug-in-Hybrid-Vehicle-Regis/qtcv-n3tc">Maryland Open Data</a></span></div>

In [52]:
client = Socrata("opendata.maryland.gov", None)

# Initialize variables
results = []
limit = 2000
offset = 0

# Loop through the dataset using the offset parameter
while True:
    batch = client.get("qtcv-n3tc", limit=limit, offset=offset)
    if not batch:
        break
    results.extend(batch)
    offset += limit

# Convert to pandas DataFrame
ev_reg_county = pd.DataFrame.from_records(results)
ev_reg_county.head()

,year_month,fuel_category,county,count
0,2020/07,Electric,ALLEGANY,29
1,2020/07,Electric,ANNE ARUNDEL,1587
2,2020/07,Electric,BALTIMORE,1508
3,2020/07,Electric,BALTIMORE CITY,770
4,2020/07,Electric,CALVERT,140


#### Clean Data
Clean County Total Registration Data

In [53]:
# Create a new dataframe to clean without changing the original dataframe
total_md_reg_counties = total_reg_zip.copy()

# Remove FY from the fiscal_year column
total_md_reg_counties['fiscal_year'] = total_md_reg_counties['fiscal_year'].str.replace('FY','')

# Rename the fiscal_year column to year
total_md_reg_counties.rename(columns = {'fiscal_year':'year'}, inplace = True)

# Set year to index
total_md_reg_counties.set_index('year', inplace=True)

# Convert all columns to numeric
total_md_reg_counties = total_md_reg_counties.apply(pd.to_numeric)

# Create a new dataframe from the sum of all columns
total_md_reg = pd.DataFrame(total_md_reg_counties.sum(axis=1))

# Rename the column to total_registrations
total_md_reg.rename(columns = {0:'total_registrations'}, inplace = True)

total_md_reg.head()

,total_registrations
year,
2010,4737336
2011,4782657
2012,4822378
2013,4813421
2014,4872481


###### Clean County EV and Plugin Registration Data

In [54]:
# Split the year_month column into year and month columns
ev_reg_county[['year', 'month']] = ev_reg_county['year_month'].str.split('/', expand=True)

# Drop the original year_month column if no longer needed
ev_reg_county.drop('year_month', axis=1, inplace=True)

# Convert the count column to an integer
ev_reg_county['count'] = ev_reg_county['count'].astype(int)

ev_reg_county.head()

,fuel_category,county,count,year,month
0,Electric,ALLEGANY,29,2020,07
1,Electric,ANNE ARUNDEL,1587,2020,07
2,Electric,BALTIMORE,1508,2020,07
3,Electric,BALTIMORE CITY,770,2020,07
4,Electric,CALVERT,140,2020,07


<div style="text-decoration: underline;">Build Maryland EV Registration Datasets by County</div>
<ol>
<li>Total MD Cumulative EV and Plugin Registrations by Month</li>
<li>MD Cumulative Electric Vehicle Registrations by Month</li>
<li>MD Cumulative Plug-in Hybrids by Month</li>
</ol>

In [55]:
# Group by county, year, and month and sum the count column
md_ev_reg_count_all_vehicles = ev_reg_county.groupby(['county', 'year', 'month'])['count'].sum().reset_index()

print("Total Electric / Plugin Vehicles: {}".format(md_ev_reg_count_all_vehicles.shape))

md_ev_reg_count_all_vehicles.head()

Total Electric / Plugin Vehicles: (1585, 4)


,county,year,month,count
0,AE,2022,06,1
1,AE,2022,07,1
2,AE,2022,08,1
3,AE,2022,09,1
4,AK,2022,08,1


In [56]:
# Group by fuel category
md_ev_reg_count_electric_vehicles = ev_reg_county[ev_reg_county['fuel_category'].isin(['Electric'])].reset_index(drop=True)

print("Electric Vehicles: {}".format(md_ev_reg_count_electric_vehicles.shape))

md_ev_reg_count_electric_vehicles.head()

Electric Vehicles: (1356, 5)


,fuel_category,county,count,year,month
0,Electric,ALLEGANY,29,2020,07
1,Electric,ANNE ARUNDEL,1587,2020,07
2,Electric,BALTIMORE,1508,2020,07
3,Electric,BALTIMORE CITY,770,2020,07
4,Electric,CALVERT,140,2020,07


In [57]:
# Group by fuel category plug-in hybrid
md_ev_reg_count_plugin_vehicles = ev_reg_county[ev_reg_county['fuel_category'].isin(['Plug-in Hybrid', 'Plug-In Hybrid'])].reset_index(drop=True)

print("Plug-in Vehicles: {}".format(md_ev_reg_count_plugin_vehicles.shape))

md_ev_reg_count_plugin_vehicles.head()

Plug-in Vehicles: (1482, 5)


,fuel_category,county,count,year,month
0,Plug-in Hybrid,ALLEGANY,32,2020,07
1,Plug-in Hybrid,ANNE ARUNDEL,1124,2020,07
2,Plug-in Hybrid,BALTIMORE,1230,2020,07
3,Plug-in Hybrid,BALTIMORE CITY,585,2020,07
4,Plug-in Hybrid,CALVERT,160,2020,07


<div style="text-decoration: underline;">Convert data set from MoM to cumulative data</div>

Function to calculate month-over-month change

In [58]:
def calculate_mom_change(df, group_col, year_col, month_col, value_col, new_col):
    # Convert the 'count' column to integer
    df[value_col] = df[value_col].astype(int)

    # Sort the DataFrame by year and month
    df = df.sort_values([group_col, year_col, month_col])

    # Reset the index before performing the calculation
    df = df.reset_index(drop=True)

    # Create a temporary DataFrame for holding the shifted values
    temp_df = df.groupby(group_col)[value_col].shift(1).fillna(0)

    # Calculate the MoM change and store it in the new column
    df[new_col] = df[value_col] - temp_df

    # Reset the index after performing the calculation
    df = df.reset_index(drop=True)

    return df

print("Done")

Done


##### Calculate Month-over-Month Change

1. Total MD Cumulative EV Registrations by Month

In [59]:
# Calculate month-over-month change for Total EVs
md_ev_reg_count_all_vehicles = calculate_mom_change(md_ev_reg_count_all_vehicles, 'county', 'year', 'month', 'count', 'MoM Change')

md_ev_reg_count_all_vehicles.head()

,county,year,month,count,MoM Change
0,AE,2022,06,1,1.0
1,AE,2022,07,1,0.0
2,AE,2022,08,1,0.0
3,AE,2022,09,1,0.0
4,AK,2022,08,1,1.0


2. MD Cumulative Electric Vehicle Registrations by Month

In [60]:
# Calculate month-over-month change for Electric Only
md_ev_reg_count_electric_vehicles = calculate_mom_change(md_ev_reg_count_electric_vehicles, 'county', 'year', 'month', 'count', 'MoM Change')

md_ev_reg_count_electric_vehicles.head()

,fuel_category,county,count,year,month,MoM Change
0,Electric,AE,1,2022,06,1.0
1,Electric,AE,1,2022,07,0.0
2,Electric,AE,1,2022,08,0.0
3,Electric,AE,1,2022,09,0.0
4,Electric,AK,2,2022,09,2.0


3. MD Cumulative Plug-in Hybrids by Month

In [61]:
# Calculate month-over-month change for Plug-in Hybrid Only
md_ev_reg_count_plugin_vehicles = calculate_mom_change(md_ev_reg_count_plugin_vehicles, 'county', 'year', 'month', 'count', 'MoM Change')

md_ev_reg_count_plugin_vehicles.head()

,fuel_category,county,count,year,month,MoM Change
0,Plug-In Hybrid,AK,1,2022,08,1.0
1,Plug-In Hybrid,AK,2,2022,09,1.0
2,Plug-In Hybrid,AK,3,2022,10,1.0
3,Plug-In Hybrid,AK,2,2022,11,-1.0
4,Plug-In Hybrid,AK,2,2022,12,0.0


##### Save Data to CSV Files

In [62]:
# Save Total EVs DataFrame to a CSV file
md_ev_reg_count_all_vehicles.to_csv(os.path.join(output_directory, 'md_ev_reg_count_all_vehicles.csv'), index=False)

# Save Electric Only DataFrame to a CSV file
md_ev_reg_count_electric_vehicles.to_csv(os.path.join(output_directory, 'md_ev_reg_count_electric_vehicles.csv'), index=False)

# Save Plug-in Hybrid Only DataFrame to a CSV file
md_ev_reg_count_plugin_vehicles.to_csv(os.path.join(output_directory, 'md_ev_reg_count_plugin_vehicles.csv'), index=False)

In [63]:
# Total count and MoM changes for both vehicle types
total_count_all = md_ev_reg_count_all_vehicles.groupby(['year', 'month']).agg({'count': 'sum', 'MoM Change': 'sum'}).reset_index()

# Total count and MoM changes for "Electric" vehicles
total_count_electric = md_ev_reg_count_electric_vehicles.groupby(['year', 'month', 'fuel_category']).agg({'count': 'sum', 'MoM Change': 'sum'}).reset_index()

# Total count and MoM changes for "Plug-in Hybrid" vehicles
total_count_plugin_hybrid = md_ev_reg_count_plugin_vehicles.groupby(['year', 'month', 'fuel_category']).agg({'count': 'sum', 'MoM Change': 'sum'}).reset_index()
print("Done")

Done


In [64]:
total_count_all.to_csv(output_directory / 'total_count_all.csv', index=False)
total_count_electric.to_csv(output_directory / 'total_count_electric.csv', index=False)
total_count_plugin_hybrid.to_csv(output_directory / 'total_count_plugin_hybrid.csv', index=False)

#### Pull Historic EV data from NREL

In [65]:
import requests
import json

# Define the API endpoint
url = "https://developer.nrel.gov/api/vehicles/v1/vehicles.json"

# Define the headers for the API request
headers = {
    "Accept": "application/json",
    "X-Api-Key": NREL_ID  # Replace with your NREL API key
}

# Make the API request
response = requests.get(url, headers=headers)

# Print the first 1000 characters of the raw response data
print(str(data)[:1000])

# Parse the response
data = response.json()

# Filter the vehicles for electric vehicles in Maryland
filtered_vehicles = [vehicle for vehicle in data if vehicle['fuel_type'] == 'ELEC' and vehicle['state'] == 'MD']

# Print the filtered vehicles
print(json.dumps(filtered_vehicles, indent=4))

[[], ['Alabama', '4,700', '3,300', '42,500', '40,500', '449,500', '500', '100', '0', '0', '4,051,000', '123,500'], ['Alaska', '1,300', '500', '7,300', '7,600', '50,100', '100', '0', '0', '0', '464,200', '31,700'], ['Arizona', '40,700', '15,500', '132,200', '51,000', '460,400', '900', '900', '0', '0', '5,395,300', '191,800'], ['Arkansas', '2,400', '1,800', '26,100', '28,700', '290,200', '300', '0', '0', '0', '2,241,600', '88,800'], ['California', '563,100', '315,300', '1,355,900', '163,600', '1,343,200', '12,600', '1,500', '11,800', '0', '30,512,600', '710,500'], ['Colorado', '37,000', '16,100', '113,600', '53,800', '346,700', '600', '100', '0', '0', '4,456,600', '208,400'], ['Connecticut', '13,300', '9,200', '55,400', '8,800', '140,700', '400', '0', '0', '0', '2,578,400', '44,300'], ['Delaware', '3,000', '2,000', '16,700', '4,100', '67,400', '100', '0', '0', '0', '796,400', '14,600'], ['District of Columbia', '3,700', '2,500', '16,100', '300', '17,400', '100', '0', '0', '0', '278,900',

TypeError: string indices must be integers

### Vehicle Sales | USA | All Vehicles
Due to limited public data on EV sales, we must use the following <a href="https://www.autosinnovate.org/resources/electric-vehicle-sales-dashboard">Autosinnovate Website</a>.
Sourcing of the data is as follows:


In [ ]:
# Build a function to pull annual vehicle sales data from the St. Louis Federal Reserve API
def get_vehicle_sales(api_key):
    # Define the endpoint URL
    url = "https://api.stlouisfed.org/fred/series/observations"

    # Define the parameters
    params = {
        "series_id": "TOTALSA",
        "api_key": api_key,
        "file_type": "json",
        "frequency": "m",  # Fetch monthly data
        "observation_start": "2020-01-01",
        "observation_end": "2023-12-31"
    }

    # Send the HTTP request
    response = requests.get(url, params=params)

    # Check if the request was successful
    if response.status_code == 200:
        # Parse the JSON response
        data = response.json()["observations"]

        # Create a dataframe from the data
        df = pd.DataFrame(data)

        # Convert 'date' column to datetime and 'value' column to float
        df['date'] = pd.to_datetime(df['date'])
        df['value'] = df['value'].astype(float)

        # Group by year and calculate the sum for each year
        df = df.groupby(df['date'].dt.year)['value'].sum().reset_index()

        # Return the dataframe
        return df
    else:
        print(f"Error occurred: {response.status_code}")
        print(response.text)

# Replace "YOUR_API_KEY" with your actual FRED API key
us_vehicle_sales = get_vehicle_sales(FRED_Key)

print(us_vehicle_sales)

#### Save the data to a csv file

In [ ]:
# Save data using the save_data() function
save_data(us_vehicle_sales, "USA_vehicle_sales.csv")

### Vehicle Sales | USA | Electric Vehicles

<p>This notebook includes data on the sales of electric vehicles (EVs) in the United States for the years 2020 to 2023. The data is gathered from multiple sources and is embedded within the text of various web pages.</p>

<p>The sales data for each year is as follows:</p>

<ul>
  <li><strong>2020</strong>: The sales of electric vehicles in 2020 were reported to be 308,000 units (<a href="https://www.energy.gov/eere/vehicles/articles/fotw-1172-january-24-2022-sales-new-light-duty-plug-electric-vehicles-united">source</a>).</li>
  <li><strong>2021</strong>: In 2021, the sales nearly doubled from the previous year, reaching a total of 608,000 units (<a href="https://www.energy.gov/eere/vehicles/articles/fotw-1172-january-24-2022-sales-new-light-duty-plug-electric-vehicles-united">source</a>).</li>
  <li><strong>2022</strong>: The EV market continued to grow in 2022, breaking records with an estimated 918,500 light electric vehicle sales (<a href="https://www.statista.com/statistics/200002/international-car-sales-since-1990/">source</a>).</li>
  <li><strong>2023</strong>: As of April 2023, the cumulative sales of EVs for the year were reported to be 401,135 units (<a href="https://www.anl.gov/es/light-duty-electric-drive-vehicles-monthly-sales-updates">source</a>).</li>
</ul>

<p>The data was manually extracted from the text of these sources and is subject to the accuracy of the reporting by these sources.</p>


In [ ]:
# Creating a dictionary with the EV sales data
data = {
    "Year": [2020, 2021, 2022, 2023],
    "EV Sales": [308000, 608000, 918500, 401135]  # Note: The 2023 data is as of April
}

# Creating a pandas DataFrame from the dictionary
df = pd.DataFrame(data)

# Set year to inded
df.set_index('Year', inplace=True)

# Displaying the DataFrame
df

#### Save the data to a csv file

In [ ]:
# Save data using the save_data() function
save_data(df, "USA_ev_sales.csv")

### Vehicle Sales | Maryland | All Vehicles

In [ ]:
# Unauthenticated client only works with public data sets. Note 'None'
# in place of application token, and no username or password:
client = Socrata("opendata.maryland.gov", None)

# Initialize variables
results = []
limit = 2000
offset = 0

# Loop through the dataset using the offset parameter
while True:
    batch = client.get("un65-7ipd", limit=limit, offset=offset)
    if not batch:
        break
    results.extend(batch)
    offset += limit

# Convert to pandas DataFrame
total_md_sales_zip = pd.DataFrame.from_records(results)
total_md_sales_zip.head()

#### Clean the data

In [ ]:
# Save new datafram
total_md_sales = total_md_sales_zip

# Convert new and used to int
total_md_sales['new'] = total_md_sales['new'].astype(int)
total_md_sales['used'] = total_md_sales['used'].astype(int)

# Sum monthly new and used sales by year
total_md_sales = total_md_sales.groupby(['year'])[['new', 'used']].sum().reset_index()

# Create a total sales column
total_md_sales['total'] = total_md_sales['new'] + total_md_sales['used']

total_md_sales

#### Save the data to a csv file

In [ ]:
# Save to csv file with the save_data() function
save_data(total_md_sales_zip, "MD_vehicle_sales.csv")

### Vehicle Sales | Maryland | Electric Vehicles

At present, it is difficult to find accurate sales data about electric vehicle sales in Maryland. As such, we will develop an estimate using the earlier vehicle sales data points and registration data.

According to the following <a href="https://www.surfky.com/electric-car-sales-maryland">source</a>, electric vehicle sales in Maryland were:
- 2016: 2185
- 2017: 3244
- 2018: 6299
- 2019: 6105
- 2020: 6033

In [ ]:
# Save the data to a dictionary
data = {
    "Year": [2016, 2017, 2018, 2019, 2020],
    "EV Sales": [2185, 3244, 6299, 6105, 6033]
}

# Create a pandas DataFrame from the dictionary
df = pd.DataFrame(data)

# Set year to index
df.set_index('Year', inplace=True)

# Create new dataframe from total_reg_zip
md_reg = total_reg_zip.copy()

# Remove FY from fiscal_year column
md_reg['fiscal_year'] = md_reg['fiscal_year'].str.replace('FY ', '')

# Convert fiscal_year to int
md_reg['fiscal_year'] = md_reg['fiscal_year'].astype(int)

# Rename fiscal_year to year
md_reg.rename(columns={'fiscal_year': 'year'}, inplace=True)

# Set year to index
md_reg.set_index('year', inplace=True)

# Convert all columns to int
md_reg = md_reg.astype(int)

# Sum columns to get total registrations
md_reg['total'] = md_reg.sum(axis=1)

# Keep total column only
md_reg = md_reg[['total']]

md_reg.head()

#### Prepare Maryland registration data and combine with EV sales data

In [ ]:
# Filter to 2015-2020
md_reg = md_reg.loc[2015:2020]

# Create a new column showing the YoY change in number of registrations
md_reg['change'] = md_reg['total'].diff()

# Remove 2015 data
md_reg = md_reg.loc[2016:2020]

md_reg.head()

In [ ]:
# Add total EV sales to the dataframe
md_reg['EV Sales'] = df['EV Sales']
md_reg.head()

In [ ]:
# Calculate the percentage of EV sales
md_reg['% EV Sales'] = md_reg['EV Sales'] / md_reg['total'] * 100
md_reg.head()

#### Save the data to a csv file

## US EV Market Analyis

### Present current state of the EV market in terms of vehicle registrations by state

#### Total US EV Registrations vs. Total US Vehicle Registrations

In [ ]:
# Edit total vehicle data frame to keep the EV column and sum the rest
total_vehicles = to[['State', 'EVs', 'Total']].groupby('State').sum().reset_index()
total_vehicles.head()

## Registration Forecast Modeling

In [ ]:
# Ensure the data has a datetime index
ev_reg_county['date'] = pd.to_datetime(ev_reg_county[['year', 'month']].assign(day=1))
ev_monthly = ev_reg_county.groupby(['fuel_category', 'date']).agg({'count': 'sum'}).reset_index()

# Pivot the data for separate time series for each fuel category
consolidated_ev_data = ev_monthly.pivot_table(index='date', columns='fuel_category', values='count').fillna(0)

if 'Plug-In Hybrid' in consolidated_ev_data.columns and 'Plug-in Hybrid' in consolidated_ev_data.columns:
    consolidated_ev_data['Total Plug-In Hybrid'] = consolidated_ev_data['Plug-In Hybrid'] + consolidated_ev_data['Plug-in Hybrid']

ev_timeseries = consolidated_ev_data['Electric']
plugin_timeseries = consolidated_ev_data['Total Plug-In Hybrid']
print(consolidated_ev_data)
print(ev_timeseries)
print(plugin_timeseries)

Combine Data Sets

In [ ]:
# Combine the EV, Plugin, electricity price, and gas price data sets
combined_data = pd.merge(consolidated_ev_data, gas_prices, left_index=True, right_on='date', how='left')

# Set the index to the date
combined_data = combined_data.set_index('date')

# Consolidate duplicate Plugin Hybrid columns
combined_data['Total Plug-In Hybrid'] = combined_data['Plug-In Hybrid'] + combined_data['Plug-in Hybrid']

# Drop the duplicate Plugin Hybrid columns and rename Total Plug-In Hybrid to Plug-In Hybrid
combined_data = combined_data.drop(columns=['Plug-In Hybrid', 'Plug-in Hybrid'])

# Rename the Electric column to EV
combined_data = combined_data.rename(columns={'Electric': 'ev_registrations'})

# Rename Total Plug-In Hybrid to plugin_hybrid
combined_data = combined_data.rename(columns={'Total Plug-In Hybrid': 'plugin_hybrid_registrations'})

# Index by date
# combined_data = combined_data.set_index('date')

# Add the electricity price data
combined_data = pd.merge(combined_data, electricity_prices, left_index=True, right_on='date', how='left')

# remove rows with missing values or nan
combined_data = combined_data.dropna()

# Set the index to the date
combined_data = combined_data.set_index('date')

combined_data
print(combined_data)

#### Support Formulas

##### Calculate CAGR

In [ ]:
# Function to calculate CAGR
def CAGR(first, last, periods):
    # Monthly periods to annual
    periods = periods / 12
    return (last / first) ** (1 / periods) - 1

##### Plot Forecast Line Connector

In [ ]:
def forecast_plt_connector(hist_df, forecast_df):
    # Get the last date of the historical data
    last_date = hist_df.index[-1]

    # Create new forecast series that includes the last date of the historical data
    new_index = pd.date_range(start=last_date, periods=len(forecast_df.index) + 1, freq='MS')
    connected_forecast = pd.Series(data=np.append(hist_df[-1], forecast_df.values), index=new_index)
    return connected_forecast

### Plot Historic Vehicle Registrations for EVs and Plug-in Hybrids

In [ ]:
# Check the data
print(combined_data.isna().sum())

In [ ]:
# Check for variation
print(combined_data.describe())

In [ ]:
# Check for collinearity
print(combined_data.corr())

In [ ]:
# Check for cointegration
from statsmodels.tsa.vector_ar.vecm import coint_johansen

johansen_test = coint_johansen(combined_data, det_order=0, k_ar_diff=1)

print(johansen_test.lr1)  # Test statistic
print(johansen_test.cvt)  # Critical values


In [ ]:
from statsmodels.tsa.vector_ar.vecm import VECM

vecm_model = VECM(combined_data, coint_rank=1)
vecm_fit = vecm_model.fit()

print(vecm_fit.summary())

In [ ]:
# Suppose 'vecm' is your fitted model, 'lagged_values' are your most recent observations
forecast = vecm_fit.predict(combined_data['ev_registrations'], steps=5)


### Prepare for Time Series Forecasting by checking for Stationarity

In [ ]:
from statsmodels.tsa.api import VAR

# Check for stationarity
model = VAR(combined_data)

# Check for stationarity
for i in range(1, 13):
    result = model.fit(i)
    print('Lag Order =', i)
    print('AIC : ', result.aic)
    print('BIC : ', result.bic)
    print('FPE : ', result.fpe)
    print('HQIC: ', result.hqic, '\n')

### Check for Multicollinearity
Multicollinearity is a phenomenon in which one predictor variable in a multiple regression model can be linearly predicted from the others with a substantial degree of accuracy.

In [ ]:
!pip install seaborn
import seaborn as sns
import matplotlib.pyplot as plt
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Create a correlation matrix
correlation_matrix = combined_data.corr()

# Use seaborn to create a heatmap of the correlation matrix
plt.figure(figsize=(12,8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

# Calculate the Variance Inflation Factor (VIF) for each variable
vif_data = pd.DataFrame()
vif_data["feature"] = combined_data.columns
vif_data["VIF"] = [variance_inflation_factor(combined_data.values, i) for i in range(len(combined_data.columns))]
print(vif_data)

### Apply the ADF Test to Check for Stationarity
The Augmented Dickey-Fuller test (ADF test) is a type of statistical test called a unit root test. The null hypothesis of the test is that the time series can be represented by a unit root, that is, it is not stationary (has some time-dependent structure).

The p-value of the ADF test is interpreted as follows:

A small p-value (typically ≤ 0.05) indicates strong evidence against the null hypothesis, so you reject the null hypothesis (which means the data is stationary).
A large p-value (> 0.05) indicates weak evidence against the null hypothesis, so you fail to reject the null hypothesis (which means the data is not stationary).


The ADF statistic needs to be compared with the critical values. If the ADF statistic is less than the critical value, you reject the null hypothesis and can say that the time series is stationary. If the ADF statistic is greater than the critical value, you fail to reject the null hypothesis and can say that the time series is non-stationary.

In [ ]:
from statsmodels.tsa.stattools import adfuller

# Apply the ADF test to each column
for name, column in combined_data.items():
    adf_result = adfuller(column)
    print(f'ADF Statistic for {name}: {adf_result[0]}')
    print(f'p-value: {adf_result[1]}')
    print(f'n_lags: {adf_result[2]}')
    print(f'Num Observations used: {adf_result[3]}')
    print('Critical Values:')
    for key, value in adf_result[4].items():
        print('\t%s: %.3f' % (key, value))
    print('\n')

### Difference the Series to Make it Stationary
Differencing the series is a way to remove the trend and make the series stationary. You can difference the series by subtracting the previous observation from the current observation. You can also difference the series by subtracting the series from itself with a time lag. The lag value is called the order of differencing. The order of differencing is the number of times you have to difference the series to make it stationary.

In [ ]:
# Differencing the series
combined_data_diff = combined_data.diff().dropna()

# Then you can check stationarity again
for name, column in combined_data_diff.items():
    adf_result = adfuller(column)
    print(f'ADF Statistic for {name}: {adf_result[0]}')
    print(f'p-value: {adf_result[1]}')
    print(f'n_lags: {adf_result[2]}')
    print(f'Num Observations used: {adf_result[3]}')
    print('Critical Values:')
    for key, value in adf_result[4].items():
        print('\t%s: %.3f' % (key, value))
    print('\n')

In [ ]:
# Second order differencing
combined_data_diff_2 = combined_data_diff.diff().dropna()

# Then check stationarity again
for name, column in combined_data_diff_2.items():
    adf_result = adfuller(column)
    print(f'ADF Statistic for {name}: {adf_result[0]}')
    print(f'p-value: {adf_result[1]}')
    print(f'n_lags: {adf_result[2]}')
    print(f'Num Observations used: {adf_result[3]}')
    print('Critical Values:')
    for key, value in adf_result[4].items():
        print('\t%s: %.3f' % (key, value))
    print('\n')

In [ ]:
# Plot the differenced series
plt.figure(figsize=(12,8))
plt.plot(combined_data_diff)
plt.title('Differenced Series')
plt.show()


In [ ]:
# Create a boxplot for each variable
plt.figure(figsize=(12,8))
sns.boxplot(data=combined_data)
plt.title('Boxplot of Variables')
plt.show()

In [ ]:
# Define the start date of the forecast period using the last date in the EV timeseries data and adding one
start_date = ev_timeseries.index[-1] + pd.DateOffset(months=1)

# Define the end date of the forecast period by adding a specified number of years to the start date
end_date = start_date + pd.DateOffset(years=5)
# end_date = '2027-12'
print(f'The forecast period goes from {start_date.year}-{start_date.month} to {end_date.year}-{end_date.month}')

In [ ]:
from statsmodels.tsa.api import VAR

model = VAR(combined_data_diff_2)
results = model.fit(5) # Use the optimal lag order found earlier or use a new one.

# To view model summary
print(results.summary())

In [ ]:
# Forecasting
lag_order = results.k_ar
forecast_input = combined_data_diff_2.values[-lag_order:]
forecast_steps = 5 # Number of steps to forecast

# Forecast
forecast_output = results.forecast(y=forecast_input, steps=forecast_steps)
print(forecast_output)

In [ ]:
# Place the forecasted output into a DataFrame
forecast_df = pd.DataFrame(forecast_output, columns=combined_data.columns)

# Reverse the second order differencing (Take a cumulative sum)
forecast_df_cumsum = forecast_df.cumsum()

# Add the last value from the original series
forecast_df_cumsum += combined_data_diff.iloc[-1]

# Now reverse the first order differencing
forecast_df_cumsum = forecast_df_cumsum.cumsum()

# Add the last value from the shifted series
forecast_df_cumsum += combined_data.iloc[-1]

print(forecast_df_cumsum)

In [ ]:
import matplotlib.pyplot as plt

# Plot historic data
plt.figure(figsize=(12,8))
for i in combined_data.columns:
    plt.plot(combined_data.index, combined_data[i], label=i)
plt.title("Historic Data")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.show()

# Plot forecasted data
# First, make sure that your forecasted DataFrame has a DateTime index
forecast_start_date = combined_data.index[-1] + pd.DateOffset(months=1)
forecast_end_date = forecast_start_date + pd.DateOffset(months=forecast_steps-1)
forecast_dates = pd.date_range(forecast_start_date, forecast_end_date, freq='M')

forecast_df_cumsum.index = forecast_dates

plt.figure(figsize=(12,8))
for i in forecast_df_cumsum.columns:
    plt.plot(forecast_df_cumsum.index, forecast_df_cumsum[i], label=i)
plt.title("Forecasted Data")
plt.xlabel("Time")
plt.ylabel("Value")
plt.legend()
plt.show()


In [ ]:
forecast_steps = 62

# Fit the VAR model and forecast
model = VAR(endog=combined_data_diff_2)
model_fit = model.fit(maxlags=12)

# Getting the last lag order number of observations
lag_order = model_fit.k_ar
last_observations = combined_data_diff_2.values[-lag_order:]

# Create the forecast
forecast = model_fit.forecast(y=last_observations, steps=forecast_steps)

# Convert forecast array into dataframe
forecast_df = pd.DataFrame(forecast, columns=combined_data_diff_2.columns)

# Undo the second differentiation (add first_diff_df's last value)
for col in forecast_df.columns:
    forecast_df[col] = forecast_df[col].cumsum() + combined_data_diff[col].iloc[-1]

# Undo the first differentiation
forecast_df_cumsum = forecast_df.cumsum() + combined_data.iloc[-1]

# Select only ev_registrations column
forecast_ev_registrations = forecast_df_cumsum['ev_registrations']

# Create a date range for the forecast data
forecast_start_date = combined_data.index[-1] + pd.DateOffset(months=1)
forecast_dates = pd.date_range(start=forecast_start_date, periods=forecast_steps, freq='M')

# Assign the new date range to the forecast
forecast_ev_registrations.index = forecast_dates

# Plot the historic and forecast results
plt.figure(figsize=(12,8))
plt.plot(combined_data['ev_registrations'], label='Historic')
plt.plot(forecast_ev_registrations, label='Forecast')
plt.legend(loc='upper left')
plt.title('EV Registrations: Historic vs Forecast')
plt.show()

In [ ]:
# Calculate CAGRs
ev_cagr = CAGR(ev_timeseries[0], ev_timeseries[-1], len(ev_timeseries) - 1)
plugin_cagr = CAGR(plugin_timeseries[0], plugin_timeseries[-1], len(plugin_timeseries) - 1)

plt.figure(figsize=(14, 7))

# Plotting the Electric Vehicles forecast
plt.plot(ev_timeseries.index, ev_timeseries, label='Registered Electric Vehicles')

# Plotting the Plug-in Hybrids forecast
plt.plot(plugin_timeseries.index, plugin_timeseries, label='Registered Plug-in Hybrids')

plt.text(0.01, 0.95, f'Historical EV CAGR: {ev_cagr * 100:.2f}%', transform=plt.gca().transAxes)
plt.text(0.01, 0.90, f'Historical Plugin Hybrid CAGR: {plugin_cagr * 100:.2f}%', transform=plt.gca().transAxes)

# Formatting the x-axis to display years only
years = mdates.YearLocator()   # Every year
years_fmt = mdates.DateFormatter('%Y')
ax = plt.gca()
ax.xaxis.set_major_locator(years)
ax.xaxis.set_major_formatter(years_fmt)

# Adding title and labels
plt.title('Historic Electric Vehicles and Plug-in Hybrids Vehicle Registrations in Maryland')
plt.xlabel('Year')
plt.ylabel('Vehicle Registrations')
plt.legend()
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

# Display the plot
plt.tight_layout()
plt.show()

## Forecasting Vehicle Registrations | Approach 1 - SARIMAX No Policy

This forecast takes a simple view of the data and assumes that the growth rate of the last year will continue into the future. This is a very simple approach and does not take into account any seasonality or other factors that may affect the growth rate of the data.

The model used to do this is the SARIMAX model from the statsmodels library. The model is fit to the data and then used to forecast the next 5 years of data. The SARIMAX model is a seasonal ARIMA model that takes into account the seasonality of the data.

Predictions are made for both the EV and Plugin Hybrid data. The predictions are then plotted against the historic data to see how well the model fits the data. The model does a good job of fitting the data, but it does not take into account any seasonality or other factors that may affect the growth rate of the data.




### Build and Prep SARIMAX model

#### Understanding SARIMAX Parameters

The `order` and `seasonal_order` parameters in the SARIMAX model refer to the orders of different components of the model. These components include:

- AR (AutoRegressive) - In the context of time series forecasting, an autoregressive model predicts future values based on past values. It's called "autoregressive" because it regresses the variable on itself. In simpler terms, it uses previous data points to predict the next data point. For instance, in an AR model of order 1 (AR(1)), the current value is based on the immediately preceding one.
- I (Integrated) - Integrated is the I in ARIMA, which stands for AutoRegressive Integrated Moving Average. It refers to the differencing of raw observations to allow for the time series to become stationary, i.e., data values are replaced by the difference between the data values and the previous values. A time series that has been made stationary through differencing is said to be integrated.
- MA (Moving Average) - In the context of time series forecasting, a moving average model uses past forecast errors in a regression-like model. It takes a set of observations, calculates the mean of those observations and uses this to forecast future values. In this model, the value of a variable at a given time is based on a linear combination of past errors.
- Seasonality - In time series analysis, seasonality refers to predictable and repeating patterns or cycles of behavior that occur over the course of the year, or over any other fixed period in the data. For instance, in retail, sales might regularly peak in December due to holiday shopping and be at their lowest in January, post-holiday season. This is a pattern that repeats every year and is thus a seasonal effect. Models like SARIMAX can handle seasonality.

For `order=(p, d, q)`, the parameters represent:
- `p`: The order of the autoregressive part (AR). This specifies the number of lags of the series to include in the AR part of the model.
- `d`: The degree of differencing (I). This specifies the number of times the series should be differenced to make it stationary.
- `q`: The order of the moving average part (MA). This specifies the number of lagged forecast errors to include in the MA part of the model.

For `seasonal_order=(P, D, Q, s)`, the parameters represent:
- `P`: The order of the seasonal part of the autoregressive model (SAR).
- `D`: The degree of seasonal differencing.
- `Q`: The order of the seasonal part of the moving average model (SMA).
- `s`: The length of the seasonal cycle.

So in the given model, `SARIMAX(df, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))`, we have:
- An AR order of 1, meaning the model uses one prior value (lag=1) in the autoregressive part.
- An I order of 1, meaning the model applies first order differencing to make the time series stationary.
- An MA order of 1, meaning the model uses one lagged forecast error in the moving average part.
- Seasonal components with the same AR, I, and MA orders of 1, with a seasonal cycle of 12 periods (indicating a yearly cycle, assuming the data is monthly).

#### Define SARIMAX No Policy Forecast Function

In [ ]:
# Function to forecast EV sales
def forecast_ev_sales(df, start_date, end_date):
    # Ensure frequency of index is monthly start (MS)
    df.index = pd.DatetimeIndex(df.index, freq='MS')

    # Calculate the first order difference of the data to make it stationary, if necessary
    df_diff = df.diff().dropna()

    # Initiate SARIMAX model with orders (1, 1, 1) for AR, I, and MA and seasonal orders (1, 1, 1, 12)
    model = SARIMAX(df, order=(1, 1, 1), seasonal_order=(1, 1, 1, 12))

    # Fit the SARIMAX model
    results = model.fit()

    # Get the forecast for the specified period, using the fitted model
    forecast = results.get_prediction(start=pd.to_datetime(start_date), end=pd.to_datetime(end_date), dynamic=False)

    # Return the predicted mean values
    return forecast.predicted_mean

# Function to run the forecast with no policy
def run_forecast_no_policy(df, end_date):
    # Define the start date as the month after the last date in the provided dataframe
    start_date = df.index[-1] + pd.DateOffset(months=1)

    # Print the forecast start date
    print(f"Forecast start date: {start_date}")

    # Generate the forecast using the forecast_ev_sales function
    forecast = forecast_ev_sales(df, start_date, end_date)

    # Return the forecast
    return forecast

#### Set Forecast Parameters

### Forecast Results for MD EVs and Plugin Hybrids

In [ ]:
# Run the forecast on EV time series data
ev_forecast_no_policy = run_forecast_no_policy(ev_timeseries, end_date)

# Run the forecast on Plugin Hybrid time series data
plugin_forecast_no_policy = run_forecast_no_policy(plugin_timeseries, end_date)

# Print the historical and forecasted EV data
print("EV Hist and Forecast")
print(ev_timeseries)
print(ev_forecast_no_policy)

# Print the historical and forecasted Plugin Hybrid data
print("Plugin Hist and Forecast")
print(plugin_timeseries)
print(plugin_forecast_no_policy)

### Plot the Forecasted EV and Plugin Hybrid Sales

#### Define Forecast Plotting Functions

##### Function to Connect the Forecasted Series to the Original Series for line plotting

In [ ]:
# Function to connect the forecasted series to the original series
def forecast_plt_connector(hist_df, forecast_df):
    # Get the last date of the historical data
    last_date = hist_df.index[-1]

    # Create new forecast series that includes the last date of the historical data
    new_index = pd.date_range(start=last_date, periods=len(forecast_df.index) + 1, freq='MS')
    connected_forecast = pd.Series(data=np.append(hist_df[-1], forecast_df.values), index=new_index)
    return connected_forecast

##### Function to line plot the Forecasted Series

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.gridspec import GridSpec

def plot_forecasts_line(forecast_dict, title):
    # Define the grid layout
    num_categories = len(forecast_dict)
    num_rows = num_categories // 3 + (num_categories % 3 > 0)
    gs = GridSpec(num_rows + 2, 3)  # +2 for the plot and increased space for table

    fig = plt.figure(figsize=(14, num_rows * 9))  # Adjust the figure size based on the number of rows
    ax = fig.add_subplot(gs[:num_rows + 1, :])  # The plot occupies the first num_rows+1 rows

    colors = ['darkblue', 'lightblue', 'darkgreen', 'lightgreen']
    color_index = 0

    for forecast_type, forecast_data in forecast_dict.items():
        timeseries = forecast_data['timeseries']
        forecast = forecast_data['forecast']

        # Plot historic data
        ax.plot(timeseries.index, timeseries, color=colors[color_index], label=f'{forecast_type} Historic')

        # Connect and plot forecast
        connected_forecast = forecast_plt_connector(timeseries, forecast)
        ax.plot(connected_forecast.index, connected_forecast, color=colors[(color_index+1)], label=f'{forecast_type} Forecast')

        # Compute and display CAGR
        connected_cagr = CAGR(timeseries[0], connected_forecast[-1], len(timeseries) + len(connected_forecast) - 1)

        # Additional descriptors
        total_added = round(connected_forecast[-1] - timeseries[0])
        descriptors = f'{forecast_type} Registrations:\nStarting of Historic: {round(timeseries[0]):,} EVs as of {timeseries.index[0].year}/{timeseries.index[0].month}\nEnd of Historic: {round(timeseries[-1]):,} EVs as of {timeseries.index[-1].year}/{timeseries.index[-1].month}\nEnd of Forecast: {round(connected_forecast[-1]):,} EVs as of {connected_forecast.index[-1].year}/{connected_forecast.index[-1].month}\nTotal vehicles added: {total_added:,}\nTotal % growth over period: {((connected_forecast[-1] - timeseries[0])/timeseries[0])*100:,.2f}%\nCAGR ({timeseries.index[0].year}/{timeseries.index[0].month} - {connected_forecast.index[-1].year}/{connected_forecast.index[-1].month}) - {connected_cagr * 100:.2f}%'

        # Create a subplot for the descriptors and add the descriptors as text to the subplot
        ax_desc = fig.add_subplot(gs[num_rows + 1 + color_index // 3, color_index % 3])
        ax_desc.text(0, 1, descriptors, horizontalalignment='left', verticalalignment='top', fontsize=10, bbox=dict(facecolor='white', alpha=0.8))
        ax_desc.axis('off')  # Hide the axes

        color_index += 2

    # Formatting the x-axis to display years only
    years = mdates.YearLocator()   # Every year
    years_fmt = mdates.DateFormatter('%Y')
    ax.xaxis.set_major_locator(years)
    ax.xaxis.set_major_formatter(years_fmt)

    # Adding title and labels
    ax.set_title(title)
    ax.set_xlabel('Year')
    ax.set_ylabel('Vehicle Registrations')
    ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    # Display the plot
    plt.tight_layout()
    plt.show()


plot_forecasts_line(forecast_data, '5-Year Forecasted Electric Vehicles and Plug-in Hybrid Registrations in Maryland (No Policy)')

#### Electric Vehicle Historic and Forecast Plot functions

In [ ]:
# Function to plot the forecasted data
def plot_forecast_bar(timeseries, forecast, type):
    # Calculate CAGRs
    timeseries_cagr = CAGR(timeseries[0], timeseries[-1], len(timeseries) - 1)
    forecast_cagr = CAGR(forecast[0], forecast[-1], len(forecast) - 1)

    # Plotting the forecasts
    plt.figure(figsize=(14, 7))

    # Plotting the historical Electric Vehicles data
    plt.bar(timeseries.index, timeseries, color='blue', label=f'Historical {type} Data', width=20)

    # Plotting the forecasted Electric Vehicles data
    plt.bar(forecast.index, forecast, color='lightblue', label=f'Forecasted {type} Data', width=20)

    # Adding CAGR text
    plt.text(0.01, 0.95, f'Historical {type} CAGR: {timeseries_cagr * 100:.2f}%', transform=plt.gca().transAxes)
    plt.text(0.01, 0.90, f'Forecasted {type} CAGR: {forecast_cagr * 100:.2f}%', transform=plt.gca().transAxes)

    # Formatting the x-axis to display years only
    years = mdates.YearLocator()   # Every year
    years_fmt = mdates.DateFormatter('%Y')
    ax = plt.gca()
    ax.xaxis.set_major_locator(years)
    ax.xaxis.set_major_formatter(years_fmt)

    # Adding title and labels
    plt.title(f'Sales for {type}s in Maryland')
    plt.xlabel('Year')
    plt.ylabel('Sales')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')

    # Display the plot
    plt.tight_layout()
    plt.show()

In [ ]:
forecast_data = {
    'Registered Electric Vehicles': {'timeseries': ev_timeseries, 'forecast': ev_forecast_no_policy},
    'Registered Plug-in Hybrids': {'timeseries': plugin_timeseries, 'forecast': plugin_forecast_no_policy},
    'Total Registered Vehicles': {'timeseries': total_timeseries, 'forecast': total_forecast_no_policy}
}

plot_forecasts_line(forecast_data, '5-Year Forecasted Electric Vehicles and Plug-in Hybrid Registrations in Maryland (No Policy)')

In [ ]:
plot_forecast_bar(ev_timeseries, ev_forecast_no_policy, 'Electric Vehicle')

In [ ]:
plot_forecast_bar(plugin_timeseries, plugin_forecast_no_policy, 'Plug-in Hybrid')

#### Electric Vehicle Historic and Forecast

In [ ]:
print("EV Historic Data")
print(ev_timeseries)
print("EV Forecast Data")
print(ev_forecast_no_policy)
print("PHV Historic Data")
print(plugin_timeseries)
print("PHV Forecast Data")
print(plugin_forecast_no_policy)

### Forecasting Policy Scenarios

#### Electric Vehicle Policy Scenario

In [ ]:
import statsmodels.api as sm
import pandas as pd

# Assuming df is your DataFrame and it has the columns mentioned above
X = df[['Ownership Tax Benefits', 'Charger Density', 'Household Disposable Income', 'Purchase Subsidies', 'Registration Tax Benefits', 'Value Added Tax Benefits', 'Charging Density', 'Road Priority', 'Waiver on Fee', 'Gasoline Price', 'Household Electricity Price']]
y = df['EV Market Share']

# Add a constant to the independent value
X = sm.add_constant(X)

# Make the Panel Regression model
model = sm.OLS(y, X)

# fit the model and print results
results = model.fit()
print(results.summary())

In [ ]:
import statsmodels.api as sm

def create_policy_impact(time_series, policy_start_date, ev_impact, phv_impact):
    policy_start_period = pd.Period(policy_start_date, freq='M')
    policy_impact = np.zeros(len(time_series))
    policy_impact[time_series.index >= policy_start_period] = 1

    return policy_impact * ev_impact, policy_impact * phv_impact

policy_start_date = '2024-01'
ev_impact = 0.20
phv_impact = 0.15

ev_policy_impact, phv_policy_impact = create_policy_impact(ev_time_series, policy_start_date, ev_impact, phv_impact)

def forecast_ev_sales_with_policy(df, exog, end_date, future_exog):
    model = sm.tsa.statespace.SARIMAX(df, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12), exog=exog)
    results = model.fit(disp=0)

    forecast_start_period = pd.Period(df.index[-1], freq='M') + 1
    forecast_end_period = pd.Period(end_date, freq='M')

    print(f"Forecast Start Period: {forecast_start_period}")
    print(f"Forecast End Period: {forecast_end_period}")

    # Select only out-of-sample exog data
    out_of_sample_exog = future_exog[-(forecast_end_period - forecast_start_period).n - 1:]  # Adjusted here

    required_periods = (forecast_end_period - forecast_start_period).n
    print(f"Required periods: {required_periods}, Provided periods: {len(out_of_sample_exog)}")

    print(f"Shape of out_of_sample_exog: {out_of_sample_exog.shape}")

    forecast = results.get_prediction(start=forecast_start_period, end=forecast_end_period, dynamic=False, exog=out_of_sample_exog)
    return forecast.predicted_mean


def create_future_exog(start_period, end_period, last_policy_impact, policy_start_date, new_policy_impact):
    # Create a new DataFrame to hold the future exog data
    future_exog = pd.DataFrame(index=pd.period_range(start=start_period, end=end_period, freq='M'))

    # Assume that the policy impact remains the same as the last known impact before the future period
    future_exog['Policy Impact'] = last_policy_impact

    policy_start_period = pd.Period(policy_start_date, freq='M')
    if policy_start_period >= start_period:
        # Calculate the remaining policy impact after the policy_start_period
        remaining_policy_impact = new_policy_impact[(policy_start_period - start_period).n:]

        # Extend the remaining_policy_impact to match the length of the future period if needed
        if len(remaining_policy_impact) < len(future_exog.loc[policy_start_period:]):
            remaining_policy_impact = np.pad(remaining_policy_impact, (0, len(future_exog.loc[policy_start_period:]) - len(remaining_policy_impact)), 'edge')

        # Assign the remaining policy impact to the future exog data
        future_exog.loc[policy_start_period:, 'Policy Impact'] = remaining_policy_impact

    # Add one additional month to the future exog array
    next_month = pd.Period(end_period + 1, freq='M')
    future_exog.loc[next_month] = future_exog.iloc[-1]


    print(f"Size of future_exog: {len(future_exog)}")
    print(future_exog)
    return future_exog

policy_start_date = '2024-01'
start_date = '2022-01'  # Update the start_date to '2022-01' instead of '2023-12'
end_date = '2027-12'


# Calculate the number of periods between start_date and end_date
start_period = pd.Period(start_date, freq='M')
end_period = pd.Period(end_date, freq='M')

# Create the future exog arrays with the correct size
ev_future_exog = create_future_exog(start_period, end_period, ev_policy_impact[-1], policy_start_date, ev_policy_impact)

electric_forecast_with_policy = forecast_ev_sales_with_policy(ev_time_series, ev_policy_impact, end_date, ev_future_exog)

print("Electric Forecast with Policy Impact:")
print(electric_forecast_with_policy)

In [ ]:
import matplotlib.pyplot as plt


historic_data = ev_time_series
forecast_with_policy = electric_forecast_with_policy

plt.figure(figsize=(10, 6))

# plot the actual data
plt.plot(historic_data, label='Actual Sales')

# plot the forecasted data
plt.plot(forecast_with_policy, label='Forecasted Sales with Policy Impact')
plt.plot(electric_forecast_series, label='Forecasted Sales with no Policy Impact')

plt.title('EV Sales Forecast with Policy Impact')
plt.xlabel('Time')
plt.ylabel('Sales')
plt.legend()

plt.show()


#### EV and PHEV Sales Forecast with Complex Policy Scenarios
In this notebook, we aim to forecast the sales of electric vehicles (EVs) and plug-in hybrid vehicles (PHEVs) while incorporating the impact of various policy scenarios. Our approach utilizes the SARIMAX model to predict future sales, accounting for policy impacts as exogenous variables.

Approach
Load and preprocess the EV and PHEV time series data.
Forecast the baseline sales of EVs and PHEVs without considering any policy impacts.
Create a function to generate policy impact arrays based on given policy start dates, durations, and impact percentages.
Develop complex policy scenarios by combining multiple policy impacts.
Modify the SARIMAX model to include policy impacts as exogenous variables.
Forecast EV and PHEV sales under the defined complex policy scenarios.
Compare the forecasts with and without policy impacts.
Policy Scenarios
We will analyze the following policy scenarios:

Scenario A: A policy is implemented starting in January 2024, lasting for 24 months. It has a 25% impact on EV sales and a 20% impact on PHEV sales.
Scenario B: A policy is implemented starting in July 2024, lasting for 36 months. It has a 15% impact on EV sales and a 10% impact on PHEV sales.
Scenario C: A policy is implemented starting in January 2025, with no specified end date. It has a 10% impact on EV sales and a 5% impact on PHEV sales.
We will create combined policy impacts for EV and PHEV sales by summing the impacts of the individual scenarios. Then, we will use the SARIMAX model to forecast sales for the complex policy scenarios and compare the results with the baseline forecasts without policy impacts.

This analysis will provide insights into the potential effects of policy interventions on EV and PHEV sales, helping policymakers and industry stakeholders make informed decisions.


In [ ]:
import statsmodels.api as sm
import warnings

# Ignore specific warnings
warnings.filterwarnings('ignore', 'Non-stationary starting autoregressive parameters')
warnings.filterwarnings('ignore', 'Non-stationary starting seasonal autoregressive')

# Your code here

# Function to create policy impact arrays
def create_policy_impact(time_series, policy_start_date, ev_impact, phv_impact):
    policy_start_period = pd.Period(policy_start_date, freq='M')
    policy_impact = np.zeros((len(time_series), 1))
    policy_impact[time_series.index >= policy_start_period] = 1

    return policy_impact * ev_impact, policy_impact * phv_impact

# Define policy impact parameters
policy_start_date = '2024-01'
ev_impact = 0.20
phv_impact = 0.15

# Calculate policy impacts for EV and PHV
ev_policy_impact, phv_policy_impact = create_policy_impact(ev_time_series, policy_start_date, ev_impact, phv_impact)

# Function to forecast EV sales with policy impact
def forecast_ev_sales_with_policy(df, exog, end_date, future_exog, time_series_start_date):
    model = sm.tsa.statespace.SARIMAX(df, exog=exog, order=(1, 1, 1), seasonal_order=(0, 1, 1, 12))
    results = model.fit()

    # Calculate the start and end periods for the forecasting range
    forecast_start_period = pd.Period(time_series_start_date, freq='M') + len(df)
    forecast_end_period = pd.Period(end_date, freq='M')

    # Perform the forecast
    forecast = results.get_prediction(start=forecast_start_period, end=forecast_end_period, dynamic=False, exog=future_exog)
    return forecast.predicted_mean

# Function to create future exogenous arrays
# Function to forecast EV sales with policy impact
def forecast_ev_sales_with_policy(df, exog, end_date, future_exog):
    model = sm.tsa.statespace.SARIMAX(df, order=(1, 1, 1), seasonal_order=(1, 1, 0, 12), exog=exog)
    results = model.fit(disp=0)

    forecast_start_period = pd.Period(df.index[-1], freq='M') + 1
    forecast_end_period = pd.Period(end_date, freq='M')

    required_periods = (forecast_end_period - forecast_start_period).n + 1
    out_of_sample_exog = future_exog[:required_periods]

    print(f"Required periods: {required_periods}, Provided periods: {len(out_of_sample_exog)}")

    forecast = results.get_prediction(start=forecast_start_period, end=forecast_end_period, dynamic=False, exog=out_of_sample_exog)
    return forecast.predicted_mean

# Define date range for forecasting
start_date = '2022-01'
end_date = '2027-12'

# Calculate the number of periods between start_date and end_date
start_period = pd.Period(start_date, freq='M')
end_period = pd.Period(end_date, freq='M')

# Create the future exog arrays with the correct size
ev_future_exog = create_future_exog(start_period, end_period, ev_policy_impact[-1], policy_start_date)
phv_future_exog = create_future_exog(start_period, end_period, phv_policy_impact[-1], policy_start_date)

# Forecast EV and PHV sales with policy impact
electric_forecast_with_policy = forecast_ev_sales_with_policy(ev_time_series, ev_policy_impact, end_date, ev_future_exog)
plugin_hybrid_forecast_with_policy = forecast_ev_sales_with_policy(plugin_time_series, phv_policy_impact, end_date, phv_future_exog)



# Define additional scenarios
scenario_a_start = '2024-01'
scenario_a_duration = 24
scenario_a_ev_impact = 0.25

scenario_a_phv_impact = 0.20

scenario_b_start = '2024-07'
scenario_b_duration = 36
scenario_b_ev_impact = 0.15
scenario_b_phv_impact = 0.10

scenario_c_start = '2025-01'
scenario_c_duration = None
scenario_c_ev_impact = 0.10
scenario_c_phv_impact = 0.05

# Create policy impact arrays for each scenario
ev_policy_impact_scenario_a, _ = create_policy_impact(ev_time_series, scenario_a_start, scenario_a_ev_impact, 0)
ev_policy_impact_scenario_b, _ = create_policy_impact(ev_time_series, scenario_b_start, scenario_b_ev_impact, 0)
ev_policy_impact_scenario_c, _ = create_policy_impact(ev_time_series, scenario_c_start, scenario_c_ev_impact, 0)

print(ev_policy_impact_scenario_a)
print(ev_policy_impact_scenario_b)

_, phv_policy_impact_scenario_a = create_policy_impact(plugin_time_series, scenario_a_start, 0, scenario_a_phv_impact)
_, phv_policy_impact_scenario_b = create_policy_impact(plugin_time_series, scenario_b_start, 0, scenario_b_phv_impact)
_, phv_policy_impact_scenario_c = create_policy_impact(plugin_time_series, scenario_c_start, 0, scenario_c_phv_impact)

# Create the future exog arrays with the correct size for each scenario
ev_future_exog_scenario_a = create_future_exog(start_period, end_period, ev_policy_impact_scenario_a[-1], scenario_a_start)
ev_future_exog_scenario_b = create_future_exog(start_period, end_period, ev_policy_impact_scenario_b[-1], scenario_b_start)
ev_future_exog_scenario_c = create_future_exog(start_period, end_period, ev_policy_impact_scenario_c[-1], scenario_c_start)

phv_future_exog_scenario_a = create_future_exog(start_period, end_period, phv_policy_impact_scenario_a[-1], scenario_a_start)
phv_future_exog_scenario_b = create_future_exog(start_period, end_period, phv_policy_impact_scenario_b[-1], scenario_b_start)
phv_future_exog_scenario_c = create_future_exog(start_period, end_period, phv_policy_impact_scenario_c[-1], scenario_c_start)

# Calculate forecasts for each scenario separately
electric_forecast_scenario_a = forecast_ev_sales_with_policy(ev_time_series, ev_policy_impact_scenario_a, end_date, ev_future_exog_scenario_a)
electric_forecast_scenario_b = forecast_ev_sales_with_policy(ev_time_series, ev_policy_impact_scenario_b, end_date, ev_future_exog_scenario_b)
electric_forecast_scenario_c = forecast_ev_sales_with_policy(ev_time_series, ev_policy_impact_scenario_c, end_date, ev_future_exog_scenario_c)

plugin_hybrid_forecast_scenario_a = forecast_ev_sales_with_policy(plugin_time_series, phv_policy_impact_scenario_a, end_date, phv_future_exog_scenario_a)
plugin_hybrid_forecast_scenario_b = forecast_ev_sales_with_policy(plugin_time_series, phv_policy_impact_scenario_b, end_date, phv_future_exog_scenario_b)
plugin_hybrid_forecast_scenario_c = forecast_ev_sales_with_policy(plugin_time_series, phv_policy_impact_scenario_c, end_date, phv_future_exog_scenario_c)

# Print forecasts for each scenario
print("Electric Forecast for Scenario A:")
print(electric_forecast_scenario_a)

print("\nPlug-in Hybrid Forecast for Scenario A:")
print(electric_forecast_scenario_b)

print("\nElectric Forecast for Scenario B:")
print(electric_forecast_scenario_c)

In [ ]:
electric_forecast_scenario_a

In [ ]:
# Combine the baseline and three scenarios into one DataFrame
combined_forecasts = pd.DataFrame({
    'scenario_a': electric_forecast_scenario_a,
    'scenario_b': electric_forecast_scenario_b,
    'scenario_c': electric_forecast_scenario_c
})

# Reset the index
combined_forecasts.reset_index(inplace=True)

# Create separate columns for year and month
combined_forecasts['year'] = combined_forecasts['index'].dt.year
combined_forecasts['month'] = combined_forecasts['index'].dt.month

# Drop the original 'index' column
combined_forecasts.drop('index', axis=1, inplace=True)

# Display the combined DataFrame
combined_forecasts

In [ ]:
# Save Electric Only DataFrame to a CSV file
combined_forecast_df.to_csv(os.path.join(output_directory, 'combined_forecast_df.csv'), index=False)

#### Import MVA vehicle registration by county
These figures represent the total number of vehicles registered in each county as of the end of each fiscal year.

In [ ]:
# MVA Vehicle Registration Counts Summarized by County for FY 2010 to FY 2022. Counts are total number of registrations 'as of' the end of each year.
path = 'https://opendata.maryland.gov/resource/kqkd-4fx8.csv'

ev_ye_reg = pd.read_csv(path)
ev_ye_reg.set_index('fiscal_year', inplace = True)
ev_ye_reg = ev_ye_reg.transpose()
ev_ye_reg = ev_ye_reg.reset_index()
ev_ye_reg.columns.name = None
ev_ye_reg.rename(columns = {'index':'County'}, inplace = True)
ev_ye_reg.head()

In [ ]:
# Filter the DataFrame for December 2022
december_2022_df = ev_reg_county[(ev_reg_county['year'] == '2022') & (ev_reg_county['month'] == '12')]

# Group by county and fuel_category, then sum the count
grouped_df = december_2022_df.groupby(['county', 'fuel_category']).agg({'count': 'sum'}).reset_index()

# Pivot the DataFrame to show fuel categories as columns
pivoted_df = grouped_df.pivot_table(index='county', columns='fuel_category', values='count', fill_value=0).reset_index()

# Calculate the total count for each county
pivoted_df['count'] = pivoted_df['Electric'] + pivoted_df['Plug-In Hybrid']

# Rename the index to 'county'
pivoted_df.index.name = 'county'

# Display the result
pivoted_df.head()

## U.S. EV Infrastructure Market

### EV Infrastructure Chargers Added by Year

In [ ]:
ev_data_us = ev_data

# Convert the 'Open Date' column to datetime format
ev_data_us['Open Date'] = pd.to_datetime(ev_data_us['Open Date'])

# Extract the year from the 'Open Date' column and create a new column called 'Open Year'
ev_data_us['Open Year'] = ev_data_us['Open Date'].dt.year

# Filter the data to only include "Public" access type
ev_data_us = ev_data_us[ev_data_us['Access Type'] == 'public']

# Define the charger categories
charger_categories = ['Level 1', 'Level 2', 'DC Fast']

# Prepare a dictionary to store the DataFrames for each category
output_dataframes = {}

# Determine the range of years to include
min_year = int(ev_data_us['Open Year'].min())
max_year = int(ev_data_us['Open Year'].max())

# Create an empty DataFrame to store the total chargers added from all three categories
total_chargers_df = pd.DataFrame(index=range(min_year, max_year + 1)).fillna(0)

# Loop through each charger category and create a chart for that category
for category in charger_categories:
    # Filter the data to only include the current charger category
    ev_data_us_cat = ev_data_us[ev_data_us[category + ' Chargers Count'] > 0]

    # Group the data by year and sum the charger counts
    cols_to_sum = [category + ' Chargers Count']
    ev_data_us_summed = ev_data_us_cat.groupby(['Open Year']).sum()[cols_to_sum]

    # Fill in missing years with zero counts
    ev_data_us_summed = ev_data_us_summed.reindex(range(min_year, max_year + 1), fill_value=0)

    # Update the total chargers DataFrame
    total_chargers_df[category] = ev_data_us_summed[cols_to_sum[0]]

    # Save the DataFrame to the dictionary
    output_dataframes[category] = ev_data_us_summed

    # Calculate the CAGR for the charger type
    if ev_data_us_summed.shape[0] >= 2:
        cagr = (ev_data_us_summed.iloc[-1] / ev_data_us_summed.iloc[0])**(1/(ev_data_us_summed.shape[0]-1)) - 1
    else:
        cagr = [0]

    # Create a bar chart for the current charger category
    fig, ax = plt.subplots()
    ev_data_us_summed.plot(kind='bar', ax=ax)
    ax.set_title('Total ' + category + ' chargers by year')
    ax.set_xlabel('Year')
    ax.set_ylabel('Total chargers')

    # Set the lower limit of the y-axis to 0
    ax.set_ylim(bottom=0)

    # Add a trendline to the bar chart
    x = ev_data_us_summed.index
    y = ev_data_us_summed[cols_to_sum[0]]
    if ev_data_us_summed.shape[0] >= 2:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        ax.plot(x, p(x), linestyle='--', linewidth=2, color='red', label='CAGR: ' + '{:.1%}'.format(cagr[0]))
    else:
        ax.plot(x, y, linestyle='--', linewidth=2, color='red', label='CAGR: N/A')

# Display the chart and legend
ax.legend()
plt.show()

# Create a bar chart for the total chargers added from all three categories together
fig, ax = plt.subplots()
total_chargers_df.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Total chargers by year (all categories)')
ax.set_xlabel('Year')
ax.set_ylabel('Total chargers')
plt.show()

# Save the total chargers DataFrame to the dictionary
output_dataframes['Total'] = total_chargers_df

# Access the DataFrame output variables for each charger category
level_1_df = output_dataframes['Level 1']
level_2_df = output_dataframes['Level 2']
dc_fast_df = output_dataframes['DC Fast']
total_df = output_dataframes['Total']

### EV Infrastructure Chargers Cumulative by Year

In [ ]:
ev_data_us = ev_data

# Convert the 'Open Date' column to datetime format
ev_data_us['Open Date'] = pd.to_datetime(ev_data_us['Open Date'])

# Extract the year from the 'Open Date' column and create a new column called 'Open Year'
ev_data_us['Open Year'] = ev_data_us['Open Date'].dt.year

# Filter the data to only include "Public" access type
ev_data_us = ev_data_us[ev_data_us['Access Type'] == 'public']

# Define the charger categories
charger_categories = ['Level 1', 'Level 2', 'DC Fast']

# Prepare a dictionary to store the DataFrames for each category
output_dataframes = {}

# Determine the range of years to include
min_year = int(ev_data_us['Open Year'].min())
max_year = int(ev_data_us['Open Year'].max())

# Create an empty DataFrame to store the total chargers added from all three categories
total_chargers_df = pd.DataFrame(index=range(min_year, max_year + 1)).fillna(0)

# Loop through each charger category and create a chart for that category
for category in charger_categories:
    # Filter the data to only include the current charger category
    ev_data_us_cat = ev_data_us[ev_data_us[category + ' Chargers Count'] > 0]

    # Group the data by year and sum the charger counts
    cols_to_sum = [category + ' Chargers Count']
    ev_data_us_summed = ev_data_us_cat.groupby(['Open Year']).sum()[cols_to_sum]

    # Fill in missing years with zero counts
    ev_data_us_summed = ev_data_us_summed.reindex(range(min_year, max_year + 1), fill_value=0)

    # Update the total chargers DataFrame
    total_chargers_df[category] = ev_data_us_summed[cols_to_sum[0]]

    # Save the DataFrame to the dictionary
    output_dataframes[category] = ev_data_us_summed

    # Calculate the cumulative sum for the current charger category
    ev_data_us_cumulative = ev_data_us_summed.cumsum()

    # Save the DataFrame to the dictionary
    output_dataframes[category + '_cumulative'] = ev_data_us_cumulative

    # Create a bar chart for the cumulative chargers
    fig, ax = plt.subplots()
    ev_data_us_cumulative.plot(kind='bar', ax=ax)
    ax.set_title('Cumulative ' + category + ' chargers by year')
    ax.set_xlabel('Year')
    ax.set_ylabel('Total chargers')

    # Set the lower limit of the y-axis to 0
    ax.set_ylim(bottom=0)

    plt.show()

# Calculate the cumulative sum for the total chargers from all categories
total_chargers_cumulative = total_chargers_df.cumsum()

# Create a bar chart for the cumulative total chargers (all categories)
fig, ax = plt.subplots()
total_chargers_cumulative.plot(kind='bar', stacked=True, ax=ax)
ax.set_title('Cumulative chargers by year (all categories)')
ax.set_xlabel('Year')
ax.set_ylabel('Total chargers')

# Set the lower limit of the y-axis to 0
ax.set_ylim(bottom=0)

plt.show()

# Save the total chargers cumulative DataFrame to the dictionary
output_dataframes['Total_cumulative'] = total_chargers_cumulative

# Access the DataFrame output variables for each charger category
level_1_cumulative_df = output_dataframes['Level 1_cumulative']
level_2_cumulative_df = output_dataframes['Level 2_cumulative']
dc_fast_cumulative_df = output_dataframes['DC Fast_cumulative']
total_cumulative_df = output_dataframes['Total_cumulative']

#### Save Dataframe to Excel File

In [ ]:
import pandas as pd
from google.colab import files
import os
import openpyxl

# Upload the existing Excel file to Google Colab
uploaded = files.upload()
existing_excel_file_name = list(uploaded.keys())[0]

# Load the existing Excel file with 'read_only=False'
book = openpyxl.load_workbook(existing_excel_file_name, read_only=False)

# Create an ExcelWriter object to save the updated Excel file
with pd.ExcelWriter(existing_excel_file_name, engine='openpyxl', mode='a') as writer:
    # Set the workbook object of the ExcelWriter to the existing Excel file
    writer.book = book

    # Add the new DataFrames as new sheets to the updated Excel file
    level_1_cumulative_df.to_excel(writer, sheet_name='Level 1 Cumulative', index=True)
    level_2_cumulative_df.to_excel(writer, sheet_name='Level 2 Cumulative', index=True)
    dc_fast_cumulative_df.to_excel(writer, sheet_name='DC Fast Cumulative', index=True)
    total_cumulative_df.to_excel(writer, sheet_name='Total Cumulative', index=True)

    # Save the updated Excel file
    writer.save()

# Download the updated Excel file to your computer
files.download(existing_excel_file_name)

# Remove the Excel file from the Google Colab environment
os.remove(existing_excel_file_name)